# Random Forest Analysis of Aerosol Optical Properties

## Overview

This notebook applies Random Forest regression to investigate the relationships between aerosol physicochemical properties and aerosol optical properties measured during the SAMBBA aircraft campaign. The analysis quantifies the relative importance of each predictor, evaluates model performance, and examines the Partial Dependence Profiles (PDPs) of the most influential variables to support the physical interpretation of the models.

## Workflow

1. Load and synchronize the processed SAMBBA datasets.
2. Merge all research flights into a single quality-controlled dataset.
3. Configure and train Random Forest regression models.
4. Evaluate model performance using an independent test dataset.
5. Rank predictor variables according to their feature importance.
6. Generate publication-quality feature importance figures and export the model performance metrics.
7. Compute and visualize Partial Dependence Profiles (PDPs) for the most influential predictors.

## Inputs

- Harmonized SAMBBA datasets generated during the preprocessing stage.
- Aerosol chemical composition, optical properties, aircraft position, and ancillary variables from the selected research flights.

## Outputs

- Random Forest regression models for the selected target variables.
- Feature importance plots.
- CSV tables containing model performance metrics.
- Publication-quality Partial Dependence Profile (PDP) figures for the most influential predictors.

## Requirements

This notebook assumes that the preprocessing workflow has already been completed and that the harmonized SAMBBA datasets are available following the repository directory structure.

In [ ]:
# Import the libraries required for data processing, statistical analysis and visualization.
import re
import numpy as np
import xarray as xr
import pandas as pd
from flights_io_commented  import get_matched_flights
from flight_processing_commented import process_flight
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestRegressor
from sklearn.linear_model import LinearRegression
from sklearn.metrics import r2_score, mean_squared_error
import matplotlib.gridspec as gridspec
from mpl_toolkits.axes_grid1.inset_locator import inset_axes
import matplotlib.pyplot as plt
from matplotlib.patches import Patch
from sklearn.inspection import permutation_importance, partial_dependence


In [ ]:
# Define the location of the SAMBBA flight archive and identify the flights to be included in the analysis.

base_dir = "/home/jovyan/SAMBBA_data/dap.ceda.ac.uk/badc/faam/data/2012"

dirs =  [
    "b732-sep-15", "b733-sep-16", 
    "b734-sep-18", "b735-sep-19", "b736-sep-19", "b737-sep-20", "b738-sep-22", 
    "b739-sep-23", "b742-sep-27", "b743-sep-27", 
    "b744-sep-28", "b745-sep-28", "b746-sep-29", "b748-oct-02", 
    "b749-oct-03", 
]

flights = get_matched_flights(base_dir, dirs)

for flight in flights:
    print(flight.core_cloud)

In [ ]:
# Process each research flight individually, synchronize measurements from the different 
# instruments and store the variables required for the combined analysis.
results = []

for flight in flights:
    out = process_flight(flight)

    # Retain only the timestamps shared by the FAAM core, AMS and SP2
    # datasets. Restricting the analysis to concurrent measurements
    # ensures that all variables describe the same atmospheric conditions.
    common_times = np.intersect1d(out.core.data_point.values, out.ams.time.values)
    # Then intersect with SP2
    common_times = np.intersect1d(common_times, out.sp2.time.values)
    
    
    core_file_matched = out.core.sel(data_point=common_times)
    ams_resampled_matched = out.ams.sel(time=common_times)
    sp2_resampled_matched = out.sp2.sel(time=common_times)
    resampled_neph_matched = out.neph.sel(time=common_times)
    Dg_aligned_matched = out.Dgs.reindex(pd.to_datetime(common_times), method="nearest")
    PM_ratio_matched = out.PM_ratio.sel(PCAS2TSPM = common_times)
    
    # Core file variables
    Height = core_file_matched['GPS_ALT'].values.squeeze()
    Lon = core_file_matched['LON_GPS'].values.squeeze()
    Lat = core_file_matched['LAT_GPS'].values.squeeze()
    CO_conc_matched = core_file_matched['CO_AERO'].values.squeeze()
    PSAP_matched = core_file_matched['PSAP_LIN'].values.squeeze()

    # Cabin conditions
    CAB_PRES = core_file_matched["CAB_PRES"].values.squeeze()      # hPa
    CAB_TEMP = core_file_matched["CAB_TEMP"].values.squeeze() + 273.15  # K

    #Neph file variables
    TSC_blue = resampled_neph_matched["TSC_BLUU"].values.squeeze()
    TSC_green = resampled_neph_matched["TSC_GRNU"].values.squeeze()
    TSC_red = resampled_neph_matched["TSC_REDU"].values.squeeze()

    # STP reference
    P_STP = 1013.25   # hPa
    T_STP = 273.15    # K

    conversion_factor = (CAB_PRES / P_STP) * (T_STP / CAB_TEMP)

    #Sp2 file variables
    BC = sp2_resampled_matched.values * conversion_factor  # Convert to ATP conditions
    #AMS file variables
    ORG = ams_resampled_matched['ORG'].values * conversion_factor  # Convert to ATP conditions
    ORG_unc = ams_resampled_matched['ORG_uncertainty'].values * conversion_factor  # Convert to ATP conditions
    SO4 = ams_resampled_matched['SO4'].values * conversion_factor  # Convert to ATP conditions
    SO4_unc = ams_resampled_matched['SO4_uncertainty'].values * conversion_factor  # Convert to ATP conditions
    NH4 = ams_resampled_matched['NH4'].values * conversion_factor  # Convert to ATP conditions
    NH4_unc = ams_resampled_matched['NH4_uncertainty'].values * conversion_factor  # Convert to ATP conditions
    NO3 = ams_resampled_matched['NO3'].values * conversion_factor  # Convert to ATP conditions
    NO3_unc = ams_resampled_matched['NO3_uncertainty'].values * conversion_factor  # Convert to ATP conditions
    AS = ams_resampled_matched['NH4_2SO4'].values * conversion_factor  # Convert to ATP conditions
    AN = ams_resampled_matched['NH4NO3'].values * conversion_factor  # Convert to ATP conditions
    #PCASP variables
    Dg = Dg_aligned_matched
    ratio_pm = PM_ratio_matched
    
    
    # Extract the flight number (b###) from the file path
    match = re.search(r'b\d{3}', flight.core)  # Looks for 'b' followed by 3 digits
    flight_number = match.group(0) if match else "Unknown"
    
    results.append({
        'ORG': ORG,
        'BC': BC,
        'SO4': SO4,
        'NH4': NH4,
        'NO3': NO3,
        'AS': AS,
        'AN': AN,
        'TSC_blue':TSC_blue,
        'TSC_green':TSC_green,
        'TSC_red':TSC_red,
        'Exctinction': PSAP_matched+TSC_green,
        'ABS_PSAP':PSAP_matched,
        'DG_aligned': Dg,
        'PM_ratio': ratio_pm,
        'CO': CO_conc_matched,
        'ALT' : Height,
        'LON': Lon,
        'LAT': Lat,
        'flight': flight_number
    })

In [ ]:
# Merge the processed observations from all flights into a single DataFrame for the statistical analyses.
# Combine all flight data
combined_data = []
for entry in results:
    try:
        df = pd.DataFrame({
            'TSC_blue': entry['TSC_blue'],
            'TSC_green': entry['TSC_green'],
            'TSC_red': entry['TSC_red'],
            'ABS_PSAP':entry['ABS_PSAP'],
            'rBC': entry['BC'],
            'SO4': entry['SO4'],
            'NH4': entry['NH4'],
            'NO3': entry['NO3'],
            'AS': entry['AS'],
            'AN': entry['AN'],
            'OA': entry['ORG'],
            'DG': entry['DG_aligned'],
            'PM_coarse': entry['PM_ratio'],
            'ALT' : entry['ALT'],
            'LON': entry['LON'],
            'LAT': entry['LAT'],
            'CO': entry['CO'],
            'Ext': entry['Exctinction'],
            'Flight': entry['flight']
        })

        combined_data.append(df)
    except Exception as e:
        print(f"Skipping flight {entry['flight']} due to error: {e}")

# Concatenate all flights
df_all = pd.concat(combined_data)
df_clean = df_all.replace([np.inf, -np.inf], np.nan).dropna()

In [ ]:

# ============================================================
# AUDIT 1 — Observational support and flight contributions
# ============================================================

# ------------------------------------------------------------
# Configuration
# ------------------------------------------------------------

variables = ["OA", "rBC"]

response_pairs = {
    "OA": {
        "response": "TSC_green",
        "label": "Scattering coefficient (Mm$^{-1}$)",
        "title": "Scattering vs. OA",
    },
    "rBC": {
        "response": "ABS_PSAP",
        "label": "Absorption coefficient (Mm$^{-1}$)",
        "title": "Absorption vs. rBC",
    },
}

# Create a copy for diagnostics
df_audit = df_clean.copy()

# ------------------------------------------------------------
# 1. Basic concentration statistics
# ------------------------------------------------------------

stats = (
    df_audit[variables]
    .describe(percentiles=[0.05, 0.25, 0.50, 0.75, 0.95])
    .T
)

print("Concentration statistics:")
display(stats)

# ------------------------------------------------------------
# 2. Observation counts by concentration bin
# ------------------------------------------------------------

for var in variables:

    print(f"\nObservation counts — {var}")

    bins = np.linspace(
        df_audit[var].min(),
        df_audit[var].max(),
        21
    )

    counts = pd.cut(
        df_audit[var],
        bins=bins,
        include_lowest=True
    ).value_counts(sort=False)

    display(counts.to_frame("N"))

# ------------------------------------------------------------
# 3. Concentration distributions
# ------------------------------------------------------------

fig, axes = plt.subplots(
    2, 2,
    figsize=(13, 8)
)

for i, var in enumerate(variables):

    values = df_audit[var].dropna()

    # Histogram
    ax = axes[i, 0]

    ax.hist(
        values,
        bins=35,
        edgecolor="black",
        alpha=0.8
    )

    for q, style, label in [
        (5, "--", "P5"),
        (50, "-", "P50"),
        (95, "--", "P95"),
    ]:
        value = np.percentile(values, q)

        ax.axvline(
            value,
            linestyle=style,
            linewidth=2,
            label=f"{label} = {value:.2f}"
        )

    ax.set_xlabel(f"{var} ($\\mu g\\ m^{{-3}}$)")
    ax.set_ylabel("Number of observations")
    ax.set_title(f"{var} distribution")
    ax.legend(frameon=False)

    # Empirical cumulative distribution
    ax = axes[i, 1]

    sorted_values = np.sort(values)
    cumulative = np.arange(1, len(values) + 1) / len(values)

    ax.plot(
        sorted_values,
        cumulative,
        linewidth=2
    )

    ax.set_xlabel(f"{var} ($\\mu g\\ m^{{-3}}$)")
    ax.set_ylabel("Cumulative fraction")
    ax.set_title(f"{var} empirical distribution")
    ax.grid(alpha=0.3)

plt.tight_layout()
plt.show()

# ------------------------------------------------------------
# 4. Measured optical response coloured by flight
# ------------------------------------------------------------

flights_list = sorted(df_audit["Flight"].dropna().unique())

cmap = plt.get_cmap("tab20")
flight_colors = {
    flight: cmap(i % 20)
    for i, flight in enumerate(flights_list)
}

fig, axes = plt.subplots(
    1, 2,
    figsize=(14, 6)
)

for ax, var in zip(axes, variables):

    response = response_pairs[var]["response"]

    for flight in flights_list:

        subset = df_audit[
            df_audit["Flight"] == flight
        ]

        ax.scatter(
            subset[var],
            subset[response] * 1e6,
            s=22,
            alpha=0.65,
            color=flight_colors[flight],
            label=flight
        )

    ax.set_xlabel(f"{var} ($\\mu g\\ m^{{-3}}$)")
    ax.set_ylabel(response_pairs[var]["label"])
    ax.set_title(response_pairs[var]["title"])
    ax.grid(alpha=0.25)

axes[1].legend(
    title="Flight",
    bbox_to_anchor=(1.02, 1),
    loc="upper left",
    frameon=False
)

plt.tight_layout()
plt.show()

# ------------------------------------------------------------
# 5. Flight-specific concentration statistics
# ------------------------------------------------------------

flight_stats = (
    df_audit
    .groupby("Flight")
    .agg(
        N=("OA", "size"),
        OA_mean=("OA", "mean"),
        OA_p95=("OA", lambda x: x.quantile(0.95)),
        OA_max=("OA", "max"),
        rBC_mean=("rBC", "mean"),
        rBC_p95=("rBC", lambda x: x.quantile(0.95)),
        rBC_max=("rBC", "max"),
    )
    .sort_values("OA_p95", ascending=False)
)

display(flight_stats)


In [ ]:
# ------------------------------------------------------------
# 1. Concentration bins
# ------------------------------------------------------------

bins_config = {
    "OA": [
        0, 5, 10, 20, 30, 50, 100, np.inf
    ],
    "rBC": [
        0, 0.2, 0.5, 1.0, 1.5, 2.0, 3.0, np.inf
    ]
}

# ------------------------------------------------------------
# 2. Calculate observation counts
# ------------------------------------------------------------

bin_results = {}

for variable, bins in bins_config.items():

    df_temp = df_audit[["Flight", variable]].copy()

    df_temp["Concentration_bin"] = pd.cut(
        df_temp[variable],
        bins=bins,
        right=False,
        include_lowest=True
    )

    counts = (
        df_temp
        .groupby("Concentration_bin", observed=False)
        .size()
        .rename("N")
        .to_frame()
    )

    counts["Percentage"] = (
        counts["N"] / len(df_temp) * 100
    )

    bin_results[variable] = counts

    print(f"\n{variable} — observation counts")
    display(counts)

# ------------------------------------------------------------
# 3. Flight composition within each bin
# ------------------------------------------------------------

flight_bin_results = {}

for variable, bins in bins_config.items():

    df_temp = df_audit[["Flight", variable]].copy()

    df_temp["Concentration_bin"] = pd.cut(
        df_temp[variable],
        bins=bins,
        right=False,
        include_lowest=True
    )

    flight_counts = pd.crosstab(
        df_temp["Concentration_bin"],
        df_temp["Flight"]
    )

    flight_bin_results[variable] = flight_counts

    print(f"\n{variable} — flight composition")
    display(flight_counts)

# ------------------------------------------------------------
# 4. Plot observation counts by concentration bin
# ------------------------------------------------------------

fig, axes = plt.subplots(
    1, 2,
    figsize=(14, 5)
)

for ax, variable in zip(axes, ["OA", "rBC"]):

    counts = bin_results[variable]

    ax.bar(
        range(len(counts)),
        counts["N"],
        edgecolor="black",
        alpha=0.8
    )

    ax.set_xticks(range(len(counts)))
    ax.set_xticklabels(
        [str(x) for x in counts.index],
        rotation=35,
        ha="right"
    )

    ax.set_xlabel(
        f"{variable} concentration bin ($\\mu g\\ m^{{-3}}$)"
    )
    ax.set_ylabel("Number of observations")
    ax.set_title(f"Observational support — {variable}")
    ax.grid(axis="y", alpha=0.25)

plt.tight_layout()
plt.show()

# ------------------------------------------------------------
# 5. Plot flight contribution by concentration bin
# ------------------------------------------------------------

for variable in ["OA", "rBC"]:

    flight_counts = flight_bin_results[variable]

    ax = flight_counts.plot(
        kind="bar",
        stacked=True,
        figsize=(12, 5),
        width=0.85
    )

    ax.set_title(
        f"Flight contribution across {variable} bins"
    )
    ax.set_xlabel(
        f"{variable} concentration bin ($\\mu g\\ m^{{-3}}$)"
    )
    ax.set_ylabel("Number of observations")
    ax.legend(
        title="Flight",
        bbox_to_anchor=(1.02, 1),
        loc="upper left"
    )

    plt.xticks(rotation=35, ha="right")
    plt.tight_layout()
    plt.show()


In [ ]:

# ============================================================
# AUDIT 3 — Leave-One-Flight-Out (LOFO) sensitivity analysis
# ============================================================

# ------------------------------------------------------------
# 1. Configuration
# ------------------------------------------------------------

N_ESTIMATORS = 600
RANDOM_STATE = 42
TEST_SIZE = 0.20
N_GRID = 60

# Relevant target-predictor combinations
analysis_cases = {
    "Scattering_vs_OA": {
        "target": "TSC_green",
        "predictor": "OA",
        "title": "Scattering response to OA",
        "ylabel": "Predicted scattering (Mm$^{-1}$)",
    },
    "Absorption_vs_rBC": {
        "target": "ABS_PSAP",
        "predictor": "rBC",
        "title": "Absorption response to rBC",
        "ylabel": "Predicted absorption (Mm$^{-1}$)",
    },
}

# ------------------------------------------------------------
# 2. Reproduce the original reduced-model predictor selection
# ------------------------------------------------------------

full_drop = [
    "Ext", "TSC_green", "TSC_blue", "TSC_red",
    "Flight", "ABS_PSAP", "DG", "AS", "AN"
]

reduced_drop = full_drop + [
    "LAT", "LON", "ALT", "CO"
]


def get_rf_features(df, target_variable):

    drop_columns = list(
        set(reduced_drop + [target_variable])
    )

    features = df.drop(columns=drop_columns)

    return features


# ------------------------------------------------------------
# 3. Train and evaluate one RF model
# ------------------------------------------------------------

def fit_rf_sensitivity_model(
    df,
    target_variable,
    n_estimators=N_ESTIMATORS,
    random_state=RANDOM_STATE,
    test_size=TEST_SIZE,
):

    features = get_rf_features(
        df,
        target_variable
    )

    # Original notebook converts optical targets to Mm-1
    target = df[target_variable] * 1e6

    X_train, X_test, y_train, y_test = train_test_split(
        features,
        target,
        test_size=test_size,
        random_state=random_state,
    )

    rf = RandomForestRegressor(
        n_estimators=n_estimators,
        random_state=random_state,
        n_jobs=-1,
    )

    rf.fit(X_train, y_train)

    y_pred = rf.predict(X_test)

    metrics = {
        "R2": r2_score(y_test, y_pred),
        "RMSE": np.sqrt(mean_squared_error(y_test, y_pred)),
        "MBE": np.mean(y_pred - y_test),
        "N_total": len(df),
        "N_train": len(X_train),
        "N_test": len(X_test),
    }

    feature_importance = pd.Series(
        rf.feature_importances_,
        index=features.columns
    ).sort_values(ascending=False)

    return (
        rf,
        X_train,
        X_test,
        y_train,
        y_test,
        metrics,
        feature_importance,
    )


# ------------------------------------------------------------
# 4. Calculate the original-style PDP
# ------------------------------------------------------------

def calculate_original_pdp(
    rf,
    X_train,
    predictor,
    n_grid=N_GRID,
):

    values = X_train[predictor].dropna()

    grid = np.linspace(
        values.min(),
        values.max(),
        n_grid
    )

    p05 = values.quantile(0.05)
    p95 = values.quantile(0.95)

    pdp_mean = []
    pdp_std = []

    for value in grid:

        X_temp = X_train.copy()

        X_temp[predictor] = value

        predictions = rf.predict(X_temp)

        pdp_mean.append(np.mean(predictions))
        pdp_std.append(np.std(predictions))

    return pd.DataFrame({
        "predictor_value": grid,
        "pdp_mean": pdp_mean,
        "pdp_std": pdp_std,
        "inside_p05_p95": (
            (grid >= p05) & (grid <= p95)
        ),
    }), p05, p95


# ------------------------------------------------------------
# 5. Identify flights
# ------------------------------------------------------------

flights_list = sorted(
    df_clean["Flight"].dropna().unique()
)

print("Flights included in the analysis:")
print(flights_list)

print("\nNumber of observations per flight:")
display(
    df_clean["Flight"]
    .value_counts()
    .sort_index()
)


# ------------------------------------------------------------
# 6. Run baseline and leave-one-flight-out models
# ------------------------------------------------------------

all_results = {}
metrics_records = []
coverage_records = []

for case_name, config in analysis_cases.items():

    target = config["target"]
    predictor = config["predictor"]

    print("\n" + "=" * 65)
    print(case_name)
    print("=" * 65)

    # Baseline plus one model excluding each flight
    exclusions = ["None"] + flights_list

    for excluded_flight in exclusions:

        if excluded_flight == "None":

            df_subset = df_clean.copy()
            model_name = "Baseline"

        else:

            df_subset = df_clean[
                df_clean["Flight"] != excluded_flight
            ].copy()

            model_name = f"Without_{excluded_flight}"

        # ----------------------------------------------------
        # Train model
        # ----------------------------------------------------

        (
            rf,
            X_train,
            X_test,
            y_train,
            y_test,
            metrics,
            feature_importance,
        ) = fit_rf_sensitivity_model(
            df=df_subset,
            target_variable=target,
        )

        # ----------------------------------------------------
        # Calculate PDP
        # ----------------------------------------------------

        pdp_df, p05, p95 = calculate_original_pdp(
            rf=rf,
            X_train=X_train,
            predictor=predictor,
        )

        # ----------------------------------------------------
        # Save model outputs
        # ----------------------------------------------------

        all_results[(case_name, model_name)] = {
            "model": rf,
            "X_train": X_train,
            "X_test": X_test,
            "metrics": metrics,
            "feature_importance": feature_importance,
            "pdp": pdp_df,
            "p05": p05,
            "p95": p95,
            "df_subset": df_subset,
        }

        # ----------------------------------------------------
        # Metrics table
        # ----------------------------------------------------

        metrics_records.append({
            "Analysis": case_name,
            "Model": model_name,
            "Excluded_flight": excluded_flight,
            **metrics,
        })

        # ----------------------------------------------------
        # Concentration coverage
        # ----------------------------------------------------

        coverage_records.append({
            "Analysis": case_name,
            "Model": model_name,
            "Excluded_flight": excluded_flight,
            "Predictor": predictor,
            "N_observations": len(df_subset),
            "Full_min": df_subset[predictor].min(),
            "Full_p05": df_subset[predictor].quantile(0.05),
            "Full_median": df_subset[predictor].median(),
            "Full_p95": df_subset[predictor].quantile(0.95),
            "Full_max": df_subset[predictor].max(),
            "Train_min": X_train[predictor].min(),
            "Train_p05": X_train[predictor].quantile(0.05),
            "Train_p95": X_train[predictor].quantile(0.95),
            "Train_max": X_train[predictor].max(),
            "PDP_grid_min": pdp_df["predictor_value"].min(),
            "PDP_grid_max": pdp_df["predictor_value"].max(),
        })

        print(
            f"{model_name:20s} | "
            f"N={len(df_subset):3d} | "
            f"R2={metrics['R2']:.3f} | "
            f"RMSE={metrics['RMSE']:.2f} | "
            f"MBE={metrics['MBE']:.2f}"
        )


# ------------------------------------------------------------
# 7. Convert outputs to DataFrames
# ------------------------------------------------------------

lofo_metrics = pd.DataFrame(metrics_records)

lofo_coverage = pd.DataFrame(coverage_records)

print("\nLOFO model performance:")
display(
    lofo_metrics.round(3)
)

print("\nLOFO concentration coverage:")
display(
    lofo_coverage.round(3)
)


In [ ]:

# ============================================================
# AUDIT 4 — Plot LOFO PDP sensitivity
# ============================================================

fig, axes = plt.subplots(
    1, 2,
    figsize=(15, 6)
)

for ax, (case_name, config) in zip(
    axes,
    analysis_cases.items()
):

    predictor = config["predictor"]

    # --------------------------------------------------------
    # Baseline
    # --------------------------------------------------------

    baseline = all_results[
        (case_name, "Baseline")
    ]

    pdp = baseline["pdp"]

    ax.plot(
        pdp["predictor_value"],
        pdp["pdp_mean"],
        color="black",
        linewidth=3,
        label="All flights",
        zorder=10,
    )

    # Baseline percentile limits
    ax.axvline(
        baseline["p05"],
        color="black",
        linestyle=":",
        alpha=0.7,
    )

    ax.axvline(
        baseline["p95"],
        color="black",
        linestyle=":",
        alpha=0.7,
    )

    # --------------------------------------------------------
    # All leave-one-flight-out curves
    # --------------------------------------------------------

    for flight in flights_list:

        model_name = f"Without_{flight}"

        result = all_results[
            (case_name, model_name)
        ]

        pdp = result["pdp"]

        if flight == "b737":
            color = "tab:red"
            linewidth = 2.5
            alpha = 1
            label = "Without b737"

        elif flight == "b742":
            color = "tab:blue"
            linewidth = 2.5
            alpha = 1
            label = "Without b742"

        else:
            color = "gray"
            linewidth = 1
            alpha = 0.45
            label = None

        ax.plot(
            pdp["predictor_value"],
            pdp["pdp_mean"],
            color=color,
            linewidth=linewidth,
            alpha=alpha,
            label=label,
        )

    ax.set_xlabel(
        f"{predictor} ($\\mu g\\ m^{{-3}}$)",
        fontsize=12
    )

    ax.set_ylabel(
        config["ylabel"],
        fontsize=12
    )

    ax.set_title(
        config["title"],
        fontsize=13
    )

    ax.grid(alpha=0.25)

    ax.legend(
        frameon=False,
        fontsize=9
    )

plt.tight_layout()
plt.show()


In [ ]:

# ============================================================
# AUDIT 5 — Focused comparison: baseline vs b737 and b742
# ============================================================

fig, axes = plt.subplots(
    1, 2,
    figsize=(14, 5)
)

for ax, (case_name, config) in zip(
    axes,
    analysis_cases.items()
):

    comparisons = [
        ("Baseline", "All flights"),
        ("Without_b737", "Without b737"),
        ("Without_b742", "Without b742"),
    ]

    for model_name, label in comparisons:

        result = all_results[
            (case_name, model_name)
        ]

        pdp = result["pdp"]

        ax.plot(
            pdp["predictor_value"],
            pdp["pdp_mean"],
            linewidth=2.5,
            label=label,
        )

    ax.set_xlabel(
        f"{config['predictor']} ($\\mu g\\ m^{{-3}}$)"
    )

    ax.set_ylabel(config["ylabel"])
    ax.set_title(config["title"])
    ax.legend(frameon=False)
    ax.grid(alpha=0.25)

plt.tight_layout()
plt.show()


In [ ]:

# ============================================================
# AUDIT 6 — RF performance sensitivity
# ============================================================

fig, axes = plt.subplots(
    1, 2,
    figsize=(14, 5)
)

for ax, case_name in zip(
    axes,
    analysis_cases.keys()
):

    subset = lofo_metrics[
        lofo_metrics["Analysis"] == case_name
    ].copy()

    ax.bar(
        subset["Model"],
        subset["R2"],
        edgecolor="black",
        alpha=0.8,
    )

    ax.set_title(case_name)
    ax.set_ylabel("Test R²")
    ax.set_ylim(
        min(0, subset["R2"].min() - 0.05),
        min(1.05, subset["R2"].max() + 0.05)
    )

    ax.tick_params(
        axis="x",
        rotation=90
    )

    ax.grid(axis="y", alpha=0.25)

plt.tight_layout()
plt.show()


In [ ]:

# ============================================================
# AUDIT 7 — Export LOFO results
# ============================================================

lofo_metrics.to_csv(
    "LOFO_RF_metrics.csv",
    index=False
)

lofo_coverage.to_csv(
    "LOFO_concentration_coverage.csv",
    index=False
)

# Export all PDP curves
pdp_export = []

for (case_name, model_name), result in all_results.items():

    temp = result["pdp"].copy()

    temp["Analysis"] = case_name
    temp["Model"] = model_name

    pdp_export.append(temp)

pdp_export = pd.concat(
    pdp_export,
    ignore_index=True
)

pdp_export.to_csv(
    "LOFO_PDP_curves.csv",
    index=False
)

print("LOFO results successfully exported.")


In [ ]:

# ============================================================
# AUDIT 8 — Direct observational sensitivity to flight removal
# ============================================================

comparisons = {
    "All flights": None,
    "Without b737": "b737",
    "Without b742": "b742",
}

response_cases = {
    "Scattering vs OA": {
        "x": "OA",
        "y": "TSC_green",
        "xlabel": "OA ($\\mu g\\ m^{-3}$)",
        "ylabel": "Scattering (Mm$^{-1}$)",
        "xlim": (0, 110),
    },
    "Absorption vs rBC": {
        "x": "rBC",
        "y": "ABS_PSAP",
        "xlabel": "rBC ($\\mu g\\ m^{-3}$)",
        "ylabel": "Absorption (Mm$^{-1}$)",
        "xlim": (0, 5.8),
    },
}

fig, axes = plt.subplots(
    2, 3,
    figsize=(17, 10)
)

for row, (case_name, config) in enumerate(
    response_cases.items()
):

    for col, (comparison_name, excluded) in enumerate(
        comparisons.items()
    ):

        ax = axes[row, col]

        if excluded is None:
            subset = df_clean.copy()
        else:
            subset = df_clean[
                df_clean["Flight"] != excluded
            ].copy()

        ax.scatter(
            subset[config["x"]],
            subset[config["y"]] * 1e6,
            alpha=0.55,
            s=24,
            edgecolors="none",
        )

        ax.set_xlim(config["xlim"])

        ax.set_xlabel(config["xlabel"])
        ax.set_ylabel(config["ylabel"])

        ax.set_title(
            f"{case_name}\n{comparison_name}"
        )

        ax.grid(alpha=0.25)

plt.tight_layout()
plt.show()


In [ ]:

# ============================================================
# AUDIT 9 — Synthetic linear control experiment
# ============================================================

# ------------------------------------------------------------
# 1. Configuration
# ------------------------------------------------------------

PREDICTORS = [
    "OA",
    "rBC",
    "SO4",
    "NH4",
    "NO3",
    "PM_coarse",
]

RF_PARAMETERS = {
    "n_estimators": 600,
    "random_state": 42,
    "n_jobs": -1,
}

TEST_SIZE = 0.20
RANDOM_STATE = 42
N_GRID = 60

synthetic_cases = {
    "Scattering_vs_OA": {
        "predictor": "OA",
        "observed_response": "TSC_green",
        "response_label": "Scattering (Mm$^{-1}$)",
        "title": "Synthetic linear control: OA",
    },
    "Absorption_vs_rBC": {
        "predictor": "rBC",
        "observed_response": "ABS_PSAP",
        "response_label": "Absorption (Mm$^{-1}$)",
        "title": "Synthetic linear control: rBC",
    },
}

# ------------------------------------------------------------
# 2. Prepare predictor matrix
# ------------------------------------------------------------

X = df_clean[PREDICTORS].copy()

# Keep only observations complete for all predictors
valid = X.notna().all(axis=1)

X = X.loc[valid].copy()
df_reference = df_clean.loc[valid].copy()

print("Number of observations:", len(X))
print("Predictors:", list(X.columns))

# ------------------------------------------------------------
# 3. Generate synthetic response
# ------------------------------------------------------------

def generate_synthetic_response(
    x,
    observed_y,
    random_state=42,
):
    """
    Generate a strictly linear response with independent,
    homoscedastic Gaussian noise.

    The linear trend and residual standard deviation are
    calibrated from the corresponding measured relationship.
    """

    x = np.asarray(x, dtype=float)
    observed_y = np.asarray(observed_y, dtype=float)

    # Fit observed linear relationship
    beta_1, beta_0 = np.polyfit(
        x,
        observed_y,
        deg=1,
    )

    observed_linear = beta_0 + beta_1 * x

    residuals = observed_y - observed_linear

    noise_std = np.std(
        residuals,
        ddof=2,
    )

    # Generate independent noise
    rng = np.random.default_rng(random_state)

    noise = rng.normal(
        loc=0,
        scale=noise_std,
        size=len(x),
    )

    # Strictly linear underlying relationship
    synthetic_y = (
        beta_0
        + beta_1 * x
        + noise
    )

    parameters = {
        "intercept": beta_0,
        "slope": beta_1,
        "noise_std": noise_std,
        "observed_residual_std": np.std(residuals, ddof=2),
    }

    return synthetic_y, parameters


# ------------------------------------------------------------
# 4. Train RF and calculate PDP
# ------------------------------------------------------------

def fit_synthetic_rf(
    X,
    y,
    predictor,
    random_state=42,
):

    X_train, X_test, y_train, y_test = train_test_split(
        X,
        y,
        test_size=TEST_SIZE,
        random_state=random_state,
    )

    rf = RandomForestRegressor(
        **RF_PARAMETERS
    )

    rf.fit(
        X_train,
        y_train,
    )

    y_pred = rf.predict(X_test)

    metrics = {
        "R2": r2_score(y_test, y_pred),
        "RMSE": np.sqrt(
            mean_squared_error(y_test, y_pred)
        ),
        "N_train": len(X_train),
        "N_test": len(X_test),
    }

    # Original-style 60-point PDP grid
    grid = np.linspace(
        X_train[predictor].min(),
        X_train[predictor].max(),
        N_GRID,
    )

    pdp_mean = []
    pdp_std = []

    for value in grid:

        X_temp = X_train.copy()

        X_temp[predictor] = value

        predictions = rf.predict(X_temp)

        pdp_mean.append(np.mean(predictions))
        pdp_std.append(np.std(predictions))

    pdp = pd.DataFrame({
        "x": grid,
        "pdp_mean": pdp_mean,
        "pdp_std": pdp_std,
    })

    p05 = X_train[predictor].quantile(0.05)
    p95 = X_train[predictor].quantile(0.95)

    return (
        rf,
        X_train,
        X_test,
        pdp,
        p05,
        p95,
        metrics,
    )


# ------------------------------------------------------------
# 5. Run both synthetic experiments
# ------------------------------------------------------------

synthetic_results = {}
synthetic_metrics = []

for case_name, config in synthetic_cases.items():

    predictor = config["predictor"]
    observed_response = config["observed_response"]

    print("\n" + "=" * 65)
    print(case_name)
    print("=" * 65)

    # Convert optical response to Mm-1,
    # following the original RF workflow
    observed_y = (
        df_reference[observed_response].to_numpy()
        * 1e6
    )

    x_values = X[predictor].to_numpy()

    # Generate synthetic target
    y_synthetic, true_parameters = (
        generate_synthetic_response(
            x=x_values,
            observed_y=observed_y,
            random_state=RANDOM_STATE,
        )
    )

    # Train RF
    (
        rf,
        X_train,
        X_test,
        pdp,
        p05,
        p95,
        metrics,
    ) = fit_synthetic_rf(
        X=X,
        y=y_synthetic,
        predictor=predictor,
        random_state=RANDOM_STATE,
    )

    # Store results
    synthetic_results[case_name] = {
        "model": rf,
        "X_train": X_train,
        "X_test": X_test,
        "y_synthetic": y_synthetic,
        "pdp": pdp,
        "p05": p05,
        "p95": p95,
        "parameters": true_parameters,
        "metrics": metrics,
    }

    synthetic_metrics.append({
        "Analysis": case_name,
        **true_parameters,
        **metrics,
    })

    print("True synthetic relationship:")
    print(
        f"Y = {true_parameters['intercept']:.3f} "
        f"+ {true_parameters['slope']:.3f} * {predictor} "
        f"+ epsilon"
    )

    print(
        f"Noise standard deviation: "
        f"{true_parameters['noise_std']:.3f}"
    )

    print("RF performance:", metrics)


synthetic_metrics = pd.DataFrame(
    synthetic_metrics
)

print("\nSynthetic model summary:")
display(synthetic_metrics.round(4))


In [ ]:

# ============================================================
# AUDIT 10 — Compare true linear response with RF-PDP
# ============================================================

fig, axes = plt.subplots(
    2, 2,
    figsize=(14, 10),
    gridspec_kw={
        "height_ratios": [2, 1]
    },
)

for col, (case_name, config) in enumerate(
    synthetic_cases.items()
):

    result = synthetic_results[case_name]

    predictor = config["predictor"]
    parameters = result["parameters"]
    pdp = result["pdp"]

    beta_0 = parameters["intercept"]
    beta_1 = parameters["slope"]

    x_grid = pdp["x"].to_numpy()

    # Known underlying linear relationship
    true_line = beta_0 + beta_1 * x_grid

    # RF-PDP
    rf_pdp = pdp["pdp_mean"].to_numpy()

    # Difference from true relationship
    difference = rf_pdp - true_line

    # --------------------------------------------------------
    # Upper panel: PDP and true relationship
    # --------------------------------------------------------

    ax = axes[0, col]

    ax.plot(
        x_grid,
        true_line,
        color="black",
        linestyle="--",
        linewidth=2.5,
        label="True linear relationship",
    )

    ax.plot(
        x_grid,
        rf_pdp,
        color="tab:blue",
        linewidth=2.5,
        label="RF-PDP",
    )

    ax.fill_between(
        x_grid,
        rf_pdp - pdp["pdp_std"].to_numpy(),
        rf_pdp + pdp["pdp_std"].to_numpy(),
        color="tab:blue",
        alpha=0.15,
        label="PDP prediction spread (±1 SD)",
    )

    ax.axvline(
        result["p05"],
        color="gray",
        linestyle=":",
        linewidth=1.5,
        label="Training P5 / P95",
    )

    ax.axvline(
        result["p95"],
        color="gray",
        linestyle=":",
        linewidth=1.5,
    )

    ax.set_title(config["title"])
    ax.set_xlabel(
        f"{predictor} ($\\mu g\\ m^{{-3}}$)"
    )
    ax.set_ylabel(config["response_label"])

    ax.grid(alpha=0.25)
    ax.legend(frameon=False, fontsize=8)

    # --------------------------------------------------------
    # Lower panel: difference from true relationship
    # --------------------------------------------------------

    ax = axes[1, col]

    ax.axhline(
        0,
        color="black",
        linestyle="--",
        linewidth=1.5,
    )

    ax.plot(
        x_grid,
        difference,
        color="tab:red",
        linewidth=2,
    )

    ax.axvline(
        result["p05"],
        color="gray",
        linestyle=":",
    )

    ax.axvline(
        result["p95"],
        color="gray",
        linestyle=":",
    )

    ax.set_xlabel(
        f"{predictor} ($\\mu g\\ m^{{-3}}$)"
    )

    ax.set_ylabel(
        "RF-PDP − true linear response"
    )

    ax.grid(alpha=0.25)

plt.tight_layout()
plt.show()


In [ ]:

# ============================================================
# AUDIT 11 — Quantify RF-PDP deviations
# ============================================================

deviation_records = []

for case_name, config in synthetic_cases.items():

    result = synthetic_results[case_name]

    predictor = config["predictor"]
    parameters = result["parameters"]
    pdp = result["pdp"]

    true_line = (
        parameters["intercept"]
        + parameters["slope"] * pdp["x"]
    )

    deviation = pdp["pdp_mean"] - true_line

    central = (
        (pdp["x"] >= result["p05"])
        & (pdp["x"] <= result["p95"])
    )

    for region_name, mask in [
        ("Full grid", np.ones(len(pdp), dtype=bool)),
        ("Training P5-P95", central.to_numpy()),
    ]:

        errors = deviation[mask]

        deviation_records.append({
            "Analysis": case_name,
            "Region": region_name,
            "MAE": np.mean(np.abs(errors)),
            "RMSE_deviation": np.sqrt(
                np.mean(errors ** 2)
            ),
            "Max_abs_deviation": np.max(
                np.abs(errors)
            ),
        })

deviation_summary = pd.DataFrame(
    deviation_records
)

display(
    deviation_summary.round(3)
)


In [ ]:

# ============================================================
# AUDIT 12 — Repeated synthetic linear controls
# ============================================================

N_SIMULATIONS = 20

repeat_records = []

for case_name, config in synthetic_cases.items():

    predictor = config["predictor"]
    observed_response = config["observed_response"]

    observed_y = (
        df_reference[observed_response].to_numpy()
        * 1e6
    )

    x_values = X[predictor].to_numpy()

    # Estimate the same underlying linear relationship
    beta_1, beta_0 = np.polyfit(
        x_values,
        observed_y,
        1,
    )

    residuals = observed_y - (
        beta_0 + beta_1 * x_values
    )

    noise_std = np.std(residuals, ddof=2)

    for simulation in range(N_SIMULATIONS):

        rng = np.random.default_rng(
            1000 + simulation
        )

        noise = rng.normal(
            0,
            noise_std,
            len(x_values),
        )

        y_synthetic = (
            beta_0
            + beta_1 * x_values
            + noise
        )

        (
            rf,
            X_train,
            X_test,
            pdp,
            p05,
            p95,
            metrics,
        ) = fit_synthetic_rf(
            X=X,
            y=y_synthetic,
            predictor=predictor,
            random_state=RANDOM_STATE,
        )

        true_line = (
            beta_0
            + beta_1 * pdp["x"]
        )

        central = (
            (pdp["x"] >= p05)
            & (pdp["x"] <= p95)
        )

        deviation = (
            pdp["pdp_mean"] - true_line
        )

        central_error = deviation[central]

        repeat_records.append({
            "Analysis": case_name,
            "Simulation": simulation,
            "R2": metrics["R2"],
            "RMSE": metrics["RMSE"],
            "PDP_MAE": np.mean(
                np.abs(central_error)
            ),
            "PDP_RMSE_deviation": np.sqrt(
                np.mean(central_error ** 2)
            ),
            "True_slope": beta_1,
            "Noise_std": noise_std,
        })


repeat_results = pd.DataFrame(
    repeat_records
)

repeat_summary = (
    repeat_results
    .groupby("Analysis")
    .agg(
        R2_mean=("R2", "mean"),
        R2_std=("R2", "std"),
        PDP_MAE_mean=("PDP_MAE", "mean"),
        PDP_MAE_std=("PDP_MAE", "std"),
        PDP_RMSE_mean=("PDP_RMSE_deviation", "mean"),
        PDP_RMSE_std=("PDP_RMSE_deviation", "std"),
    )
)

print("Repeated synthetic control summary:")
display(repeat_summary.round(3))


In [ ]:

# ============================================================
# AUDIT 13 — Export synthetic control results
# ============================================================

synthetic_metrics.to_csv(
    "Synthetic_linear_RF_metrics.csv",
    index=False,
)

deviation_summary.to_csv(
    "Synthetic_linear_PDP_deviation.csv",
    index=False,
)

repeat_results.to_csv(
    "Synthetic_linear_repeated_simulations.csv",
    index=False,
)

print("Synthetic control results exported.")


In [ ]:

# ============================================================
# AUDIT 14 — Fit measured and synthetic RF models for ALE
# ============================================================

# ------------------------------------------------------------
# 1. Configuration
# ------------------------------------------------------------

PREDICTORS = [
    "OA", "rBC", "SO4", "NH4", "NO3", "PM_coarse"
]

RF_PARAMETERS = {
    "n_estimators": 600,
    "random_state": 42,
    "n_jobs": -1,
}

TEST_SIZE = 0.20
RANDOM_STATE = 42

N_PDP_GRID = 60
N_ALE_BINS = 20

cases = {
    "Scattering_vs_OA": {
        "predictor": "OA",
        "target": "TSC_green",
        "ylabel": "Scattering (Mm$^{-1}$)",
        "title": "Scattering response to OA",
    },
    "Absorption_vs_rBC": {
        "predictor": "rBC",
        "target": "ABS_PSAP",
        "ylabel": "Absorption (Mm$^{-1}$)",
        "title": "Absorption response to rBC",
    },
}

# ------------------------------------------------------------
# 2. Prepare complete predictor dataset
# ------------------------------------------------------------

missing_columns = [
    col for col in PREDICTORS
    if col not in df_clean.columns
]

if missing_columns:
    raise ValueError(
        f"Missing predictor columns: {missing_columns}"
    )

df_model = df_clean.copy()

valid = (
    df_model[PREDICTORS].notna().all(axis=1)
    & df_model["TSC_green"].notna()
    & df_model["ABS_PSAP"].notna()
)

df_model = df_model.loc[valid].copy()

X = df_model[PREDICTORS].copy()

print("Observations used:", len(X))
print("Predictors:", PREDICTORS)

# ------------------------------------------------------------
# 3. Generate synthetic targets
# ------------------------------------------------------------

def make_synthetic_target(
    x,
    observed_y,
    random_state=42,
):
    """
    Generate a strictly linear response plus independent,
    homoscedastic Gaussian noise.

    The slope, intercept and noise SD are calibrated from
    the measured predictor-response relationship.
    """

    x = np.asarray(x, dtype=float)
    observed_y = np.asarray(observed_y, dtype=float)

    slope, intercept = np.polyfit(x, observed_y, 1)

    linear_component = intercept + slope * x

    residuals = observed_y - linear_component

    noise_sd = np.std(residuals, ddof=2)

    rng = np.random.default_rng(random_state)

    noise = rng.normal(
        loc=0,
        scale=noise_sd,
        size=len(x),
    )

    synthetic_y = linear_component + noise

    parameters = {
        "intercept": intercept,
        "slope": slope,
        "noise_sd": noise_sd,
    }

    return synthetic_y, parameters


# ------------------------------------------------------------
# 4. Fit measured and synthetic RF models
# ------------------------------------------------------------

model_results = {}

for case_name, config in cases.items():

    predictor = config["predictor"]
    target = config["target"]

    # Optical coefficients converted to Mm-1
    y_measured = (
        df_model[target].to_numpy(dtype=float) * 1e6
    )

    # Synthetic response
    y_synthetic, synthetic_parameters = (
        make_synthetic_target(
            x=X[predictor].to_numpy(),
            observed_y=y_measured,
            random_state=RANDOM_STATE,
        )
    )

    model_results[case_name] = {}

    for response_type, y in [
        ("Measured", y_measured),
        ("Synthetic", y_synthetic),
    ]:

        X_train, X_test, y_train, y_test = train_test_split(
            X,
            y,
            test_size=TEST_SIZE,
            random_state=RANDOM_STATE,
        )

        rf = RandomForestRegressor(
            **RF_PARAMETERS
        )

        rf.fit(X_train, y_train)

        y_pred = rf.predict(X_test)

        metrics = {
            "R2": r2_score(y_test, y_pred),
            "RMSE": np.sqrt(
                mean_squared_error(y_test, y_pred)
            ),
            "N_train": len(X_train),
            "N_test": len(X_test),
        }

        model_results[case_name][response_type] = {
            "model": rf,
            "X_train": X_train,
            "X_test": X_test,
            "y_train": y_train,
            "y_test": y_test,
            "metrics": metrics,
            "synthetic_parameters": synthetic_parameters,
        }

        print(
            f"{case_name} | {response_type} | "
            f"R2={metrics['R2']:.3f} | "
            f"RMSE={metrics['RMSE']:.2f}"
        )

print("\nSynthetic linear coefficients:")
for case_name, result in model_results.items():
    print(
        case_name,
        result["Synthetic"]["synthetic_parameters"]
    )


In [ ]:

# ============================================================
# AUDIT 15 — Original PDP and first-order ALE functions
# ============================================================

def calculate_pdp(
    model,
    X_train,
    feature,
    n_grid=60,
):
    """
    Original-style Partial Dependence Plot.
    """

    values = X_train[feature].dropna()

    grid = np.linspace(
        values.min(),
        values.max(),
        n_grid,
    )

    means = []
    spreads = []

    for value in grid:

        X_temp = X_train.copy()
        X_temp[feature] = value

        predictions = model.predict(X_temp)

        means.append(np.mean(predictions))
        spreads.append(np.std(predictions))

    return pd.DataFrame({
        "x": grid,
        "effect": means,
        "spread": spreads,
    })


def calculate_ale(
    model,
    X_train,
    feature,
    n_bins=20,
):
    """
    First-order Accumulated Local Effects (ALE).

    Steps:
    1. Define quantile-based bins.
    2. Select observations within each bin.
    3. Predict at the upper and lower bin boundaries.
    4. Average local prediction differences.
    5. Accumulate the local effects.
    6. Center the ALE curve using bin frequencies.

    Returns bin centers, centered ALE effects, bin counts,
    and the bin boundaries.
    """

    values = X_train[feature].dropna()

    # Quantile-based bin boundaries
    quantiles = np.linspace(
        0,
        1,
        n_bins + 1,
    )

    edges = np.unique(
        values.quantile(quantiles).to_numpy()
    )

    if len(edges) < 3:
        raise ValueError(
            f"Not enough unique values for ALE: {feature}"
        )

    # Assign observations to bins
    bin_ids = np.searchsorted(
        edges[1:-1],
        X_train[feature].to_numpy(),
        side="right",
    )

    local_effects = []
    counts = []
    centers = []

    # Calculate local effects
    for bin_index in range(len(edges) - 1):

        lower = edges[bin_index]
        upper = edges[bin_index + 1]

        mask = bin_ids == bin_index

        X_bin = X_train.loc[mask].copy()

        count = len(X_bin)

        counts.append(count)

        centers.append(
            (lower + upper) / 2
        )

        if count == 0:
            local_effects.append(np.nan)
            continue

        X_lower = X_bin.copy()
        X_upper = X_bin.copy()

        X_lower[feature] = lower
        X_upper[feature] = upper

        pred_lower = model.predict(X_lower)
        pred_upper = model.predict(X_upper)

        local_difference = (
            pred_upper - pred_lower
        )

        local_effects.append(
            np.mean(local_difference)
        )

    local_effects = np.asarray(
        local_effects,
        dtype=float,
    )

    counts = np.asarray(counts)

    # Accumulate local effects
    accumulated = np.cumsum(
        np.nan_to_num(local_effects, nan=0.0)
    )

    # Center using observation-weighted bin effects
    valid = (
        (counts > 0)
        & np.isfinite(accumulated)
    )

    weighted_mean = np.average(
        accumulated[valid],
        weights=counts[valid],
    )

    ale_centered = accumulated - weighted_mean

    ale_df = pd.DataFrame({
        "x": centers,
        "effect": ale_centered,
        "local_effect": local_effects,
        "count": counts,
        "lower": edges[:-1],
        "upper": edges[1:],
    })

    return ale_df


# ------------------------------------------------------------
# 3. Calculate PDP and ALE for all models
# ------------------------------------------------------------

effect_results = {}

for case_name, config in cases.items():

    feature = config["predictor"]

    effect_results[case_name] = {}

    for response_type in ["Measured", "Synthetic"]:

        result = model_results[case_name][response_type]

        model = result["model"]
        X_train = result["X_train"]

        pdp = calculate_pdp(
            model=model,
            X_train=X_train,
            feature=feature,
            n_grid=N_PDP_GRID,
        )

        ale = calculate_ale(
            model=model,
            X_train=X_train,
            feature=feature,
            n_bins=N_ALE_BINS,
        )

        p05 = X_train[feature].quantile(0.05)
        p95 = X_train[feature].quantile(0.95)

        effect_results[case_name][response_type] = {
            "PDP": pdp,
            "ALE": ale,
            "P05": p05,
            "P95": p95,
        }

        print(
            f"{case_name} | {response_type} | "
            f"ALE bins={len(ale)} | "
            f"Empty bins={(ale['count'] == 0).sum()}"
        )


In [ ]:

# ============================================================
# AUDIT 16 — Measured response: PDP vs ALE
# ============================================================

fig, axes = plt.subplots(
    2, 2,
    figsize=(14, 10),
)

for row, (case_name, config) in enumerate(cases.items()):

    result = effect_results[case_name]["Measured"]

    pdp = result["PDP"]
    ale = result["ALE"]

    p05 = result["P05"]
    p95 = result["P95"]

    feature = config["predictor"]

    # --------------------------------------------------------
    # PDP
    # --------------------------------------------------------

    ax = axes[row, 0]

    ax.plot(
        pdp["x"],
        pdp["effect"],
        color="tab:blue",
        linewidth=2.5,
    )

    ax.fill_between(
        pdp["x"],
        pdp["effect"] - pdp["spread"],
        pdp["effect"] + pdp["spread"],
        color="tab:blue",
        alpha=0.15,
    )

    ax.axvline(
        p05,
        color="gray",
        linestyle=":",
    )

    ax.axvline(
        p95,
        color="gray",
        linestyle=":",
    )

    ax.set_title(
        f"{config['title']} — PDP"
    )

    ax.set_xlabel(
        f"{feature} ($\\mu g\\ m^{{-3}}$)"
    )

    ax.set_ylabel(config["ylabel"])
    ax.grid(alpha=0.25)

    # --------------------------------------------------------
    # ALE
    # --------------------------------------------------------

    ax = axes[row, 1]

    ax.plot(
        ale["x"],
        ale["effect"],
        color="tab:orange",
        marker="o",
        markersize=4,
        linewidth=2,
    )

    ax.axvline(
        p05,
        color="gray",
        linestyle=":",
    )

    ax.axvline(
        p95,
        color="gray",
        linestyle=":",
    )

    ax.axhline(
        0,
        color="black",
        linestyle="--",
        linewidth=1,
    )

    ax.set_title(
        f"{config['title']} — ALE"
    )

    ax.set_xlabel(
        f"{feature} ($\\mu g\\ m^{{-3}}$)"
    )

    ax.set_ylabel(
        "Centered accumulated local effect"
    )

    ax.grid(alpha=0.25)

plt.tight_layout()
plt.show()


In [ ]:

# ============================================================
# AUDIT 17 — Synthetic response: PDP vs ALE
# ============================================================

fig, axes = plt.subplots(
    2, 2,
    figsize=(14, 10),
)

for row, (case_name, config) in enumerate(cases.items()):

    result = effect_results[case_name]["Synthetic"]
    model_result = model_results[case_name]["Synthetic"]

    pdp = result["PDP"]
    ale = result["ALE"]

    p05 = result["P05"]
    p95 = result["P95"]

    feature = config["predictor"]

    parameters = model_result["synthetic_parameters"]

    intercept = parameters["intercept"]
    slope = parameters["slope"]

    X_train = model_result["X_train"]

    # --------------------------------------------------------
    # PDP: known linear response
    # --------------------------------------------------------

    true_pdp = (
        intercept + slope * pdp["x"]
    )

    ax = axes[row, 0]

    ax.plot(
        pdp["x"],
        true_pdp,
        color="black",
        linestyle="--",
        linewidth=2.5,
        label="True linear relationship",
    )

    ax.plot(
        pdp["x"],
        pdp["effect"],
        color="tab:blue",
        linewidth=2.5,
        label="RF-PDP",
    )

    ax.axvline(
        p05,
        color="gray",
        linestyle=":",
    )

    ax.axvline(
        p95,
        color="gray",
        linestyle=":",
    )

    ax.set_title(
        f"{config['title']} — Synthetic PDP"
    )

    ax.set_xlabel(
        f"{feature} ($\\mu g\\ m^{{-3}}$)"
    )

    ax.set_ylabel(config["ylabel"])
    ax.legend(frameon=False)
    ax.grid(alpha=0.25)

    # --------------------------------------------------------
    # ALE: centered known linear relationship
    # --------------------------------------------------------

    # Center the true linear function using the training
    # predictor mean, matching the ALE zero-centered scale.
    x_reference = X_train[feature].mean()

    true_ale = (
        slope * (ale["x"] - x_reference)
    )

    ax = axes[row, 1]

    ax.plot(
        ale["x"],
        true_ale,
        color="black",
        linestyle="--",
        linewidth=2.5,
        label="True centered linear effect",
    )

    ax.plot(
        ale["x"],
        ale["effect"],
        color="tab:orange",
        marker="o",
        markersize=4,
        linewidth=2,
        label="RF-ALE",
    )

    ax.axvline(
        p05,
        color="gray",
        linestyle=":",
    )

    ax.axvline(
        p95,
        color="gray",
        linestyle=":",
    )

    ax.axhline(
        0,
        color="black",
        linewidth=0.8,
    )

    ax.set_title(
        f"{config['title']} — Synthetic ALE"
    )

    ax.set_xlabel(
        f"{feature} ($\\mu g\\ m^{{-3}}$)"
    )

    ax.set_ylabel(
        "Centered accumulated local effect"
    )

    ax.legend(frameon=False)
    ax.grid(alpha=0.25)

plt.tight_layout()
plt.show()


In [ ]:

# ============================================================
# AUDIT 18 — Compare PDP and ALE slopes
# ============================================================

slope_records = []

for case_name, config in cases.items():

    feature = config["predictor"]

    for response_type in ["Measured", "Synthetic"]:

        result = effect_results[case_name][response_type]

        pdp = result["PDP"]
        ale = result["ALE"]

        p05 = result["P05"]
        p95 = result["P95"]

        # ----------------------------------------------------
        # PDP slope inside P5-P95
        # ----------------------------------------------------

        pdp_mask = (
            (pdp["x"] >= p05)
            & (pdp["x"] <= p95)
        )

        pdp_slope, pdp_intercept = np.polyfit(
            pdp.loc[pdp_mask, "x"],
            pdp.loc[pdp_mask, "effect"],
            1,
        )

        # ----------------------------------------------------
        # ALE slope inside P5-P95
        # ----------------------------------------------------

        ale_mask = (
            (ale["x"] >= p05)
            & (ale["x"] <= p95)
            & (ale["count"] > 0)
        )

        ale_slope, ale_intercept = np.polyfit(
            ale.loc[ale_mask, "x"],
            ale.loc[ale_mask, "effect"],
            1,
        )

        # ----------------------------------------------------
        # Known slope for synthetic response
        # ----------------------------------------------------

        if response_type == "Synthetic":

            true_slope = (
                model_results[case_name]["Synthetic"]
                ["synthetic_parameters"]["slope"]
            )

        else:
            true_slope = np.nan

        slope_records.append({
            "Analysis": case_name,
            "Response": response_type,
            "PDP_slope": pdp_slope,
            "ALE_slope": ale_slope,
            "True_synthetic_slope": true_slope,
            "P05": p05,
            "P95": p95,
            "N_ALE_bins_used": ale_mask.sum(),
        })


slope_comparison = pd.DataFrame(
    slope_records
)

display(
    slope_comparison.round(4)
)


In [ ]:

# ============================================================
# AUDIT 19 — ALE bin observational support
# ============================================================

fig, axes = plt.subplots(
    2, 2,
    figsize=(14, 8),
)

for row, (case_name, config) in enumerate(cases.items()):

    feature = config["predictor"]

    for col, response_type in enumerate(
        ["Measured", "Synthetic"]
    ):

        ale = effect_results[case_name][response_type]["ALE"]

        ax = axes[row, col]

        ax.bar(
            ale["x"],
            ale["count"],
            width=(
                ale["upper"] - ale["lower"]
            ) * 0.8,
            edgecolor="black",
            alpha=0.8,
        )

        ax.set_title(
            f"{case_name} — {response_type}"
        )

        ax.set_xlabel(
            f"{feature} ($\\mu g\\ m^{{-3}}$)"
        )

        ax.set_ylabel("Training observations per bin")
        ax.grid(axis="y", alpha=0.25)

plt.tight_layout()
plt.show()


In [ ]:

# ============================================================
# AUDIT 20 — Export PDP and ALE results
# ============================================================

# Export slope comparison
slope_comparison.to_csv(
    "PDP_ALE_slope_comparison.csv",
    index=False,
)

# Export all effect curves
effect_export = []

for case_name, case_results in effect_results.items():

    for response_type, effects in case_results.items():

        for method in ["PDP", "ALE"]:

            temp = effects[method].copy()

            temp["Analysis"] = case_name
            temp["Response"] = response_type
            temp["Method"] = method

            effect_export.append(temp)

effect_export = pd.concat(
    effect_export,
    ignore_index=True,
)

effect_export.to_csv(
    "PDP_ALE_effect_curves.csv",
    index=False,
)

# Export ALE bin counts
ale_counts_export = []

for case_name, case_results in effect_results.items():

    for response_type, effects in case_results.items():

        temp = effects["ALE"][
            ["x", "lower", "upper", "count"]
        ].copy()

        temp["Analysis"] = case_name
        temp["Response"] = response_type

        ale_counts_export.append(temp)

ale_counts_export = pd.concat(
    ale_counts_export,
    ignore_index=True,
)

ale_counts_export.to_csv(
    "ALE_bin_observation_counts.csv",
    index=False,
)

print("PDP-ALE results successfully exported.")


In [ ]:

# ============================================================
# AUDIT 21 — Central-range ALE (P5-P95)
# ============================================================

def calculate_ale_central(
    model,
    X_train,
    feature,
    n_bins=10,
):
    """
    Calculate first-order ALE restricted to the training
    P5-P95 interval.

    This avoids very wide extreme-tail bins and focuses
    the interpretation on the concentration range used
    for the original PDP fitting.
    """

    values = X_train[feature].dropna()

    p05 = values.quantile(0.05)
    p95 = values.quantile(0.95)

    # Restrict observations to central range
    X_central = X_train.loc[
        (X_train[feature] >= p05)
        & (X_train[feature] <= p95)
    ].copy()

    central_values = X_central[feature]

    # Quantile-based edges within central range
    quantiles = np.linspace(
        0,
        1,
        n_bins + 1,
    )

    edges = np.unique(
        central_values.quantile(quantiles).to_numpy()
    )

    if len(edges) < 3:
        raise ValueError(
            f"Insufficient unique ALE boundaries for {feature}"
        )

    local_effects = []
    counts = []
    centers = []

    for i in range(len(edges) - 1):

        lower = edges[i]
        upper = edges[i + 1]

        if i == len(edges) - 2:
            mask = (
                (central_values >= lower)
                & (central_values <= upper)
            )
        else:
            mask = (
                (central_values >= lower)
                & (central_values < upper)
            )

        X_bin = X_central.loc[mask].copy()

        count = len(X_bin)

        counts.append(count)
        centers.append((lower + upper) / 2)

        if count == 0:
            local_effects.append(np.nan)
            continue

        X_lower = X_bin.copy()
        X_upper = X_bin.copy()

        X_lower[feature] = lower
        X_upper[feature] = upper

        local_difference = (
            model.predict(X_upper)
            - model.predict(X_lower)
        )

        local_effects.append(
            np.mean(local_difference)
        )

    local_effects = np.asarray(
        local_effects,
        dtype=float,
    )

    counts = np.asarray(counts)

    # Accumulate local effects
    accumulated = np.cumsum(
        np.nan_to_num(local_effects, nan=0.0)
    )

    valid = (
        (counts > 0)
        & np.isfinite(accumulated)
    )

    # Center the ALE curve
    weighted_mean = np.average(
        accumulated[valid],
        weights=counts[valid],
    )

    ale_centered = accumulated - weighted_mean

    return pd.DataFrame({
        "x": centers,
        "effect": ale_centered,
        "local_effect": local_effects,
        "count": counts,
        "lower": edges[:-1],
        "upper": edges[1:],
    })


In [ ]:

# ============================================================
# AUDIT 22 — Recalculate central ALE for all models
# ============================================================

central_ale_results = {}

for case_name, config in cases.items():

    feature = config["predictor"]

    central_ale_results[case_name] = {}

    for response_type in ["Measured", "Synthetic"]:

        result = model_results[case_name][response_type]

        central_ale = calculate_ale_central(
            model=result["model"],
            X_train=result["X_train"],
            feature=feature,
            n_bins=10,
        )

        central_ale_results[case_name][response_type] = (
            central_ale
        )

        print(
            f"{case_name} | {response_type} | "
            f"Bins: {len(central_ale)} | "
            f"Minimum count: {central_ale['count'].min()}"
        )


In [ ]:

# ============================================================
# AUDIT 23 — Central-range ALE comparison
# ============================================================

fig, axes = plt.subplots(
    2, 2,
    figsize=(14, 9),
)

for row, (case_name, config) in enumerate(cases.items()):

    feature = config["predictor"]

    for col, response_type in enumerate(
        ["Measured", "Synthetic"]
    ):

        ale = central_ale_results[
            case_name
        ][response_type]

        ax = axes[row, col]

        ax.plot(
            ale["x"],
            ale["effect"],
            marker="o",
            linewidth=2,
            label="Central-range ALE",
        )

        ax.axhline(
            0,
            color="black",
            linestyle="--",
            linewidth=1,
        )

        ax.set_title(
            f"{case_name} — {response_type}"
        )

        ax.set_xlabel(
            f"{feature} ($\\mu g\\ m^{{-3}}$)"
        )

        ax.set_ylabel(
            "Centered accumulated local effect"
        )

        ax.grid(alpha=0.25)
        ax.legend(frameon=False)

plt.tight_layout()
plt.show()


In [ ]:

# ============================================================
# AUDIT 24 — Central ALE slope comparison
# ============================================================

central_slope_records = []

for case_name, config in cases.items():

    feature = config["predictor"]

    for response_type in ["Measured", "Synthetic"]:

        ale = central_ale_results[
            case_name
        ][response_type]

        ale_slope, ale_intercept = np.polyfit(
            ale["x"],
            ale["effect"],
            1,
        )

        if response_type == "Synthetic":
            true_slope = (
                model_results[case_name]["Synthetic"]
                ["synthetic_parameters"]["slope"]
            )
        else:
            true_slope = np.nan

        central_slope_records.append({
            "Analysis": case_name,
            "Response": response_type,
            "Central_ALE_slope": ale_slope,
            "True_synthetic_slope": true_slope,
            "N_bins": len(ale),
            "N_observations": ale["count"].sum(),
        })

central_slope_comparison = pd.DataFrame(
    central_slope_records
)

display(
    central_slope_comparison.round(4)
)


In [ ]:

# ============================================================
# AUDIT 25 — ALE sensitivity to number of bins
# ============================================================

bin_options = [5, 10, 15, 20]

ale_bin_sensitivity = {}
ale_bin_slope_records = []

for case_name, config in cases.items():

    feature = config["predictor"]

    ale_bin_sensitivity[case_name] = {}

    for response_type in ["Measured", "Synthetic"]:

        result = model_results[case_name][response_type]

        model = result["model"]
        X_train = result["X_train"]

        ale_bin_sensitivity[case_name][response_type] = {}

        for n_bins in bin_options:

            ale = calculate_ale_central(
                model=model,
                X_train=X_train,
                feature=feature,
                n_bins=n_bins,
            )

            slope, intercept = np.polyfit(
                ale["x"],
                ale["effect"],
                1,
            )

            ale_bin_sensitivity[
                case_name
            ][response_type][n_bins] = ale

            true_slope = np.nan

            if response_type == "Synthetic":
                true_slope = (
                    model_results[case_name]["Synthetic"]
                    ["synthetic_parameters"]["slope"]
                )

            ale_bin_slope_records.append({
                "Analysis": case_name,
                "Response": response_type,
                "N_bins": n_bins,
                "ALE_slope": slope,
                "True_synthetic_slope": true_slope,
                "Min_bin_count": ale["count"].min(),
                "Max_bin_count": ale["count"].max(),
            })


ale_bin_slope_comparison = pd.DataFrame(
    ale_bin_slope_records
)

display(
    ale_bin_slope_comparison.round(3)
)


In [ ]:

# ============================================================
# AUDIT 26 — Plot ALE bin sensitivity
# ============================================================

fig, axes = plt.subplots(
    2, 2,
    figsize=(14, 10),
)

for row, (case_name, config) in enumerate(cases.items()):

    feature = config["predictor"]

    for col, response_type in enumerate(
        ["Measured", "Synthetic"]
    ):

        ax = axes[row, col]

        for n_bins in bin_options:

            ale = ale_bin_sensitivity[
                case_name
            ][response_type][n_bins]

            ax.plot(
                ale["x"],
                ale["effect"],
                marker="o",
                markersize=3,
                linewidth=1.8,
                label=f"{n_bins} bins",
            )

        ax.axhline(
            0,
            color="black",
            linestyle="--",
            linewidth=0.8,
        )

        ax.set_title(
            f"{case_name} — {response_type}"
        )

        ax.set_xlabel(
            f"{feature} ($\\mu g\\ m^{{-3}}$)"
        )

        ax.set_ylabel(
            "Centered accumulated local effect"
        )

        ax.grid(alpha=0.25)
        ax.legend(frameon=False)

plt.tight_layout()
plt.show()


In [ ]:
# ============================================================
# AUDIT 27 — Quantify ALE slope sensitivity to binning
# ============================================================

sensitivity_summary = (
    ale_bin_slope_comparison
    .groupby(["Analysis", "Response"])
    .agg(
        mean_slope=("ALE_slope", "mean"),
        std_slope=("ALE_slope", "std"),
        min_slope=("ALE_slope", "min"),
        max_slope=("ALE_slope", "max"),
    )
    .reset_index()
)

sensitivity_summary["CV_percent"] = (
    sensitivity_summary["std_slope"]
    / sensitivity_summary["mean_slope"].abs()
    * 100
)

display(sensitivity_summary.round(3))

In [ ]:

# ============================================================
# AUDIT 28 — RF sensitivity to predictor composition
# ============================================================


# ------------------------------------------------------------
# 1. Configuration
# ------------------------------------------------------------

predictor_sets = {
    "M1_Full": ["OA", "rBC", "SO4", "NH4", "NO3"],
    "M2_SO4": ["OA", "rBC", "SO4"],
    "M3_NH4": ["OA", "rBC", "NH4"],
    "M4_NO3": ["OA", "rBC", "NO3"],
    "M5_Carbonaceous": ["OA", "rBC"],
}

responses = {
    "Scattering": "TSC_green",
    "Absorption": "ABS_PSAP",
}

# Convert optical coefficients from m-1 to Mm-1
# Keep this conversion consistent with your previous RF workflow.
response_scale = {
    "Scattering": 1e6,
    "Absorption": 1e6,
}

rf_parameters = {
    "n_estimators": 600,
    "random_state": 42,
    "n_jobs": -1,
}

# ------------------------------------------------------------
# 2. Prepare a common dataset
# ------------------------------------------------------------

all_predictors = sorted(
    set(
        variable
        for variables in predictor_sets.values()
        for variable in variables
    )
)

all_responses = list(responses.values())

df_rf = df_clean[
    all_predictors + all_responses
].copy()

df_rf = df_rf.replace([np.inf, -np.inf], np.nan)
df_rf = df_rf.dropna()

print("Common dataset size:", len(df_rf))

# ------------------------------------------------------------
# 3. Generate one common train/test split
# ------------------------------------------------------------

indices = np.arange(len(df_rf))

train_idx, test_idx = train_test_split(
    indices,
    test_size=0.20,
    random_state=42,
)

print("Training observations:", len(train_idx))
print("Testing observations:", len(test_idx))


In [ ]:

# ============================================================
# AUDIT 29 — Train and evaluate reduced RF models
# ============================================================

rf_models = {}
rf_metrics = []
rf_importance = []

for response_name, response_variable in responses.items():

    print(f"\nProcessing: {response_name}")

    y = (
        df_rf[response_variable].values
        * response_scale[response_name]
    )

    y_train = y[train_idx]
    y_test = y[test_idx]

    rf_models[response_name] = {}

    for model_name, predictors in predictor_sets.items():

        X = df_rf[predictors]

        X_train = X.iloc[train_idx]
        X_test = X.iloc[test_idx]

        model = RandomForestRegressor(
            **rf_parameters
        )

        model.fit(X_train, y_train)

        y_pred = model.predict(X_test)

        r2 = r2_score(y_test, y_pred)

        rmse = np.sqrt(
            mean_squared_error(y_test, y_pred)
        )

        mbe = np.mean(y_pred - y_test)

        rf_models[response_name][model_name] = {
            "model": model,
            "predictors": predictors,
            "X_train": X_train,
            "X_test": X_test,
            "y_train": y_train,
            "y_test": y_test,
            "y_pred": y_pred,
        }

        rf_metrics.append({
            "Response": response_name,
            "Model": model_name,
            "N_predictors": len(predictors),
            "R2_test": r2,
            "RMSE_test": rmse,
            "MBE_test": mbe,
        })

        # Impurity-based feature importance
        for variable, importance in zip(
            predictors,
            model.feature_importances_,
        ):

            rf_importance.append({
                "Response": response_name,
                "Model": model_name,
                "Variable": variable,
                "MDI_importance": importance,
            })

        print(
            f"{model_name}: "
            f"R2={r2:.3f}, "
            f"RMSE={rmse:.2f}"
        )


rf_metrics_df = pd.DataFrame(rf_metrics)

rf_importance_df = pd.DataFrame(rf_importance)

display(rf_metrics_df.round(3))


In [ ]:

# ============================================================
# AUDIT 30 — Permutation importance
# ============================================================

permutation_results = []

for response_name in responses:

    for model_name in predictor_sets:

        result = rf_models[response_name][model_name]

        model = result["model"]
        X_test = result["X_test"]
        y_test = result["y_test"]

        perm = permutation_importance(
            model,
            X_test,
            y_test,
            n_repeats=30,
            random_state=42,
            scoring="r2",
            n_jobs=-1,
        )

        for i, variable in enumerate(
            result["predictors"]
        ):

            permutation_results.append({
                "Response": response_name,
                "Model": model_name,
                "Variable": variable,
                "Permutation_mean": perm.importances_mean[i],
                "Permutation_std": perm.importances_std[i],
            })


permutation_df = pd.DataFrame(
    permutation_results
)

display(permutation_df.round(4))


In [ ]:

# ============================================================
# AUDIT 31 — Manual PDP calculation
# Compatible with older scikit-learn versions
# ============================================================

pdp_results = {}

n_grid = 60

for response_name in responses:

    pdp_results[response_name] = {}

    for feature in ["OA", "rBC"]:

        pdp_results[response_name][feature] = {}

        # ----------------------------------------------------
        # Common concentration grid from training observations
        # ----------------------------------------------------

        full_X_train = df_rf.iloc[train_idx][all_predictors]

        lower = full_X_train[feature].quantile(0.05)
        upper = full_X_train[feature].quantile(0.95)

        common_grid = np.linspace(
            lower,
            upper,
            n_grid,
        )

        # ----------------------------------------------------
        # Calculate PDP for each model
        # ----------------------------------------------------

        for model_name in predictor_sets:

            result = rf_models[response_name][model_name]

            if feature not in result["predictors"]:
                continue

            model = result["model"]

            X_train = result["X_train"].copy()

            pdp_values = []

            for concentration in common_grid:

                X_modified = X_train.copy()

                # Replace the selected predictor
                X_modified[feature] = concentration

                # Predict and average
                predictions = model.predict(X_modified)

                pdp_values.append(
                    np.mean(predictions)
                )

            pdp_results[
                response_name
            ][feature][model_name] = {
                "x": common_grid.copy(),
                "y": np.asarray(pdp_values),
            }

        print(
            f"Completed PDP: {response_name} — {feature}"
        )

print("PDP calculations completed.")


In [ ]:

# ============================================================
# AUDIT 32 — Compare PDPs across predictor configurations
# ============================================================

fig, axes = plt.subplots(
    2, 2,
    figsize=(15, 10),
)

plot_config = [
    ("Scattering", "OA", axes[0, 0]),
    ("Scattering", "rBC", axes[0, 1]),
    ("Absorption", "OA", axes[1, 0]),
    ("Absorption", "rBC", axes[1, 1]),
]

for response_name, feature, ax in plot_config:

    for model_name, result in (
        pdp_results[response_name][feature].items()
    ):

        ax.plot(
            result["x"],
            result["y"],
            linewidth=2,
            label=model_name,
        )

    ax.set_title(
        f"{response_name} response to {feature}"
    )

    ax.set_xlabel(
        f"{feature} ($\\mu g\\ m^{{-3}}$)"
    )

    ax.set_ylabel(
        f"Predicted {response_name.lower()} "
        "($Mm^{-1}$)"
    )

    ax.grid(alpha=0.25)
    ax.legend(frameon=False)

plt.tight_layout()
plt.show()


In [ ]:

# ============================================================
# AUDIT 33 — MDI feature importance comparison
# ============================================================

for response_name in responses:

    fig, axes = plt.subplots(
        1, len(predictor_sets),
        figsize=(17, 4),
        sharey=True,
    )

    for ax, model_name in zip(
        axes,
        predictor_sets,
    ):

        subset = rf_importance_df[
            (rf_importance_df["Response"] == response_name)
            &
            (rf_importance_df["Model"] == model_name)
        ].sort_values(
            "MDI_importance",
            ascending=True,
        )

        ax.barh(
            subset["Variable"],
            subset["MDI_importance"],
        )

        ax.set_title(model_name)
        ax.set_xlabel("MDI importance")
        ax.grid(axis="x", alpha=0.25)

    fig.suptitle(
        f"{response_name} — Feature importance"
    )

    plt.tight_layout()
    plt.show()


In [ ]:

# ============================================================
# AUDIT 34 — PDP mean and between-tree spread
# ============================================================
pdp_spread_results = {}

n_grid = 60

for response_name in responses:

    pdp_spread_results[response_name] = {}

    for feature in ["OA", "rBC"]:

        pdp_spread_results[response_name][feature] = {}

        # Use the same training-set percentile boundaries
        full_X_train = df_rf.iloc[train_idx][all_predictors]

        p5 = full_X_train[feature].quantile(0.05)
        p95 = full_X_train[feature].quantile(0.95)

        common_grid = np.linspace(p5, p95, n_grid)

        for model_name in predictor_sets:

            result = rf_models[response_name][model_name]

            if feature not in result["predictors"]:
                continue

            model = result["model"]
            X_train = result["X_train"]

            pdp_mean = []
            pdp_std = []

            for concentration in common_grid:

                X_modified = X_train.copy()
                X_modified[feature] = concentration

                # Prediction from each individual tree
                tree_predictions = np.array([
                    tree.predict(X_modified)
                    for tree in model.estimators_
                ])

                # First average across observations for each tree
                tree_pdp = tree_predictions.mean(axis=1)

                # Then calculate ensemble mean and spread
                pdp_mean.append(tree_pdp.mean())
                pdp_std.append(tree_pdp.std(ddof=1))

            pdp_spread_results[
                response_name
            ][feature][model_name] = {
                "x": common_grid.copy(),
                "mean": np.asarray(pdp_mean),
                "std": np.asarray(pdp_std),
                "p5": p5,
                "p95": p95,
            }

print("PDP mean and spread calculations completed.")


In [ ]:

# ============================================================
# AUDIT 35 — Weighted PDP linear fits
# ============================================================

import statsmodels.api as sm

slope_results = []

for response_name in responses:

    for feature in ["OA", "rBC"]:

        for model_name, result in (
            pdp_spread_results[response_name][feature].items()
        ):

            x = result["x"]
            y = result["mean"]
            sigma = result["std"]

            # ------------------------------------------------
            # Select fitting interval
            # ------------------------------------------------

            p5 = result["p5"]
            p95 = result["p95"]

            fit_mask = (
                (x >= p5) &
                (x <= p95) &
                np.isfinite(y) &
                np.isfinite(sigma) &
                (sigma > 0)
            )

            x_fit = x[fit_mask]
            y_fit = y[fit_mask]
            sigma_fit = sigma[fit_mask]

            # ------------------------------------------------
            # Inverse-variance weights
            # ------------------------------------------------

            weights = 1.0 / sigma_fit**2

            # Weighted least squares
            X_design = sm.add_constant(x_fit)

            wls_model = sm.WLS(
                y_fit,
                X_design,
                weights=weights,
            )

            fit = wls_model.fit()

            intercept = fit.params[0]
            slope = fit.params[1]
            slope_se = fit.bse[1]

            # ------------------------------------------------
            # Goodness of fit
            # ------------------------------------------------

            y_pred = fit.predict(X_design)

            residuals = y_fit - y_pred

            ss_res = np.sum(residuals**2)
            ss_tot = np.sum(
                (y_fit - np.mean(y_fit))**2
            )

            r_squared = (
                1 - ss_res / ss_tot
                if ss_tot > 0 else np.nan
            )

            # ------------------------------------------------
            # Save results
            # ------------------------------------------------

            slope_results.append({

                "Response": response_name,
                "Predictor": feature,
                "Model": model_name,

                "Slope_m2_g": slope,
                "Slope_SE": slope_se,

                "Intercept": intercept,
                "R2_fit": r_squared,

                "Training_P5": p5,
                "Training_P95": p95,

                "Fit_min": x_fit.min(),
                "Fit_max": x_fit.max(),

                "N_grid_points": len(x_fit),

                "Mean_PDP_spread": np.mean(sigma_fit),

                "Min_PDP_spread": np.min(sigma_fit),
                "Max_PDP_spread": np.max(sigma_fit),

            })


slope_results_df = pd.DataFrame(slope_results)

display(
    slope_results_df.round(4)
)


In [ ]:

# ============================================================
# AUDIT 36 — PDP curves and weighted linear fits
# ============================================================

fig, axes = plt.subplots(
    2, 2,
    figsize=(15, 11),
)

plot_config = [
    ("Scattering", "OA", axes[0, 0]),
    ("Scattering", "rBC", axes[0, 1]),
    ("Absorption", "OA", axes[1, 0]),
    ("Absorption", "rBC", axes[1, 1]),
]

for response_name, feature, ax in plot_config:

    for model_name, result in (
        pdp_spread_results[response_name][feature].items()
    ):

        x = result["x"]
        y = result["mean"]
        sigma = result["std"]

        # Retrieve corresponding fitted slope
        fit_row = slope_results_df[
            (slope_results_df["Response"] == response_name)
            &
            (slope_results_df["Predictor"] == feature)
            &
            (slope_results_df["Model"] == model_name)
        ].iloc[0]

        slope = fit_row["Slope_m2_g"]
        intercept = fit_row["Intercept"]

        # Plot PDP mean
        ax.plot(
            x,
            y,
            linewidth=2,
            label=f"{model_name}: {slope:.2f} m² g⁻¹",
        )

        # Plot ±1 SD between-tree spread
        ax.fill_between(
            x,
            y - sigma,
            y + sigma,
            alpha=0.08,
        )

        # Plot fitted line
        ax.plot(
            x,
            intercept + slope*x,
            linestyle="--",
            alpha=0.65,
        )

    ax.set_title(
        f"{response_name} response to {feature}"
    )

    ax.set_xlabel(
        f"{feature} ($\\mu g\\ m^{{-3}}$)"
    )

    ax.set_ylabel(
        f"Predicted {response_name.lower()} "
        "($Mm^{-1}$)"
    )

    ax.grid(alpha=0.25)
    ax.legend(fontsize=8, frameon=False)

plt.tight_layout()
plt.show()


In [ ]:

# ============================================================
# AUDIT 37 — Optical efficiency sensitivity comparison
# ============================================================

relationships = [
    ("Scattering", "OA"),
    ("Scattering", "rBC"),
    ("Absorption", "OA"),
    ("Absorption", "rBC"),
]

fig, axes = plt.subplots(
    2, 2,
    figsize=(13, 9),
)

for (response_name, feature), ax in zip(
    relationships,
    axes.flat,
):

    subset = slope_results_df[
        (slope_results_df["Response"] == response_name)
        &
        (slope_results_df["Predictor"] == feature)
    ].copy()

    ax.errorbar(
        subset["Model"],
        subset["Slope_m2_g"],
        yerr=subset["Slope_SE"],
        fmt="o",
        capsize=4,
        markersize=7,
    )

    ax.set_title(
        f"{response_name} vs {feature}"
    )

    ax.set_ylabel(
        "Effective optical efficiency (m² g⁻¹)"
    )

    ax.tick_params(
        axis="x",
        rotation=35,
    )

    ax.grid(axis="y", alpha=0.25)

plt.tight_layout()
plt.show()


In [ ]:

# ============================================================
# AUDIT 38 — Relative slope changes
# ============================================================

slope_comparison = []

for response_name, feature in relationships:

    subset = slope_results_df[
        (slope_results_df["Response"] == response_name)
        &
        (slope_results_df["Predictor"] == feature)
    ].copy()

    baseline = subset.loc[
        subset["Model"] == "M1_Full",
        "Slope_m2_g",
    ].iloc[0]

    for _, row in subset.iterrows():

        relative_change = (
            (row["Slope_m2_g"] - baseline)
            / baseline * 100
            if baseline != 0 else np.nan
        )

        slope_comparison.append({

            "Response": response_name,
            "Predictor": feature,
            "Model": row["Model"],

            "Slope": row["Slope_m2_g"],
            "Slope_SE": row["Slope_SE"],

            "Relative_change_percent": relative_change,

            "R2_fit": row["R2_fit"],
            "Fit_min": row["Fit_min"],
            "Fit_max": row["Fit_max"],

        })


slope_comparison_df = pd.DataFrame(
    slope_comparison
)

display(
    slope_comparison_df.round(3)
)

# Save results
slope_results_df.to_csv(
    "RF_PDP_slope_results.csv",
    index=False,
)

slope_comparison_df.to_csv(
    "RF_PDP_slope_sensitivity.csv",
    index=False,
)


In [ ]:

# ============================================================
# AUDIT 39 — First-order Accumulated Local Effects (ALE)
# ============================================================



def calculate_ale_1d(
    model,
    X,
    feature,
    n_bins=20,
    min_samples=5,
):
    """
    Calculate first-order ALE for a single feature.

    Parameters
    ----------
    model : fitted sklearn model
        Random Forest regression model.

    X : pandas.DataFrame
        Training predictor dataset.

    feature : str
        Predictor to analyse.

    n_bins : int
        Desired number of quantile-based intervals.

    min_samples : int
        Minimum observations required in an interval.

    Returns
    -------
    ale_df : pandas.DataFrame
        ALE values, interval boundaries, and sample counts.
    """

    X = X.copy()

    # Remove missing and infinite values
    X = X.replace([np.inf, -np.inf], np.nan)
    X = X.dropna(subset=[feature])

    values = X[feature].to_numpy()

    # --------------------------------------------------------
    # 1. Define quantile-based intervals
    # --------------------------------------------------------

    quantiles = np.linspace(0, 1, n_bins + 1)

    edges = np.unique(
        np.quantile(values, quantiles)
    )

    if len(edges) < 3:
        raise ValueError(
            f"Not enough unique values for {feature}."
        )

    # --------------------------------------------------------
    # 2. Calculate local effects
    # --------------------------------------------------------

    local_effects = []
    bin_counts = []
    bin_centers = []
    bin_lower = []
    bin_upper = []

    for i in range(len(edges) - 1):

        lower = edges[i]
        upper = edges[i + 1]

        # Observations belonging to this interval
        if i == len(edges) - 2:
            mask = (
                (values >= lower) &
                (values <= upper)
            )
        else:
            mask = (
                (values >= lower) &
                (values < upper)
            )

        X_bin = X.loc[mask].copy()

        n = len(X_bin)

        if n < min_samples:
            continue

        # Create lower and upper boundary datasets
        X_lower = X_bin.copy()
        X_upper = X_bin.copy()

        X_lower[feature] = lower
        X_upper[feature] = upper

        # Predictions at both boundaries
        pred_lower = model.predict(X_lower)
        pred_upper = model.predict(X_upper)

        # Mean local prediction difference
        local_effect = np.mean(
            pred_upper - pred_lower
        )

        local_effects.append(local_effect)
        bin_counts.append(n)

        bin_centers.append(
            np.mean(X_bin[feature])
        )

        bin_lower.append(lower)
        bin_upper.append(upper)

    # --------------------------------------------------------
    # 3. Accumulate local effects
    # --------------------------------------------------------

    local_effects = np.asarray(local_effects)
    bin_counts = np.asarray(bin_counts)

    ale_values = np.cumsum(local_effects)

    # --------------------------------------------------------
    # 4. Center ALE around zero
    # --------------------------------------------------------

    weighted_mean = np.average(
        ale_values,
        weights=bin_counts,
    )

    ale_values = ale_values - weighted_mean

    # --------------------------------------------------------
    # 5. Create output dataframe
    # --------------------------------------------------------

    ale_df = pd.DataFrame({

        "feature": feature,
        "bin_lower": bin_lower,
        "bin_upper": bin_upper,
        "bin_center": bin_centers,

        "ALE": ale_values,
        "local_effect": local_effects,

        "n_observations": bin_counts,

    })

    return ale_df


In [ ]:

# ============================================================
# AUDIT 40 — Calculate ALE for selected RF models
# ============================================================

models_to_compare = [
    "M1_Full",
    "M5_Carbonaceous",
]

relationships = [
    ("Scattering", "OA"),
    ("Scattering", "rBC"),
    ("Absorption", "OA"),
    ("Absorption", "rBC"),
]

ale_results = {}

for response_name, feature in relationships:

    ale_results[(response_name, feature)] = {}

    for model_name in models_to_compare:

        result = rf_models[response_name][model_name]

        model = result["model"]
        X_train = result["X_train"]

        # Verify that the feature is available
        if feature not in X_train.columns:
            print(
                f"Skipping {response_name} - {feature} "
                f"for {model_name}: predictor unavailable."
            )
            continue

        ale_df = calculate_ale_1d(
            model=model,
            X=X_train,
            feature=feature,
            n_bins=20,
            min_samples=5,
        )

        ale_results[
            (response_name, feature)
        ][model_name] = ale_df

        print(
            f"{response_name} vs {feature} | "
            f"{model_name}: "
            f"{len(ale_df)} intervals, "
            f"{ale_df['n_observations'].sum()} observations"
        )

print("\nALE calculations completed.")


In [ ]:

# ============================================================
# AUDIT 41 — Compare PDP and ALE
# ============================================================

fig, axes = plt.subplots(
    4, 2,
    figsize=(15, 18),
)

model_colors = {
    "M1_Full": "tab:blue",
    "M5_Carbonaceous": "tab:orange",
}

relationship_labels = {
    ("Scattering", "OA"): "Scattering response to OA",
    ("Scattering", "rBC"): "Scattering response to rBC",
    ("Absorption", "OA"): "Absorption response to OA",
    ("Absorption", "rBC"): "Absorption response to rBC",
}

for row, (response_name, feature) in enumerate(relationships):

    ax_pdp = axes[row, 0]
    ax_ale = axes[row, 1]

    for model_name in models_to_compare:

        color = model_colors[model_name]

        # ----------------------------------------------------
        # PDP
        # ----------------------------------------------------

        pdp_data = pdp_spread_results[
            response_name
        ][feature][model_name]

        x_pdp = pdp_data["x"]
        y_pdp = pdp_data["mean"]
        spread = pdp_data["std"]

        ax_pdp.plot(
            x_pdp,
            y_pdp,
            color=color,
            linewidth=2,
            label=model_name,
        )

        ax_pdp.fill_between(
            x_pdp,
            y_pdp - spread,
            y_pdp + spread,
            color=color,
            alpha=0.12,
        )

        # ----------------------------------------------------
        # ALE
        # ----------------------------------------------------

        ale_df = ale_results[
            (response_name, feature)
        ][model_name]

        ax_ale.plot(
            ale_df["bin_center"],
            ale_df["ALE"],
            marker="o",
            markersize=4,
            linewidth=2,
            color=color,
            label=model_name,
        )

    # --------------------------------------------------------
    # Formatting
    # --------------------------------------------------------

    title = relationship_labels[
        (response_name, feature)
    ]

    ax_pdp.set_title(title + " — PDP")
    ax_ale.set_title(title + " — ALE")

    ax_pdp.set_xlabel(
        f"{feature} ($\\mu g\\ m^{{-3}}$)"
    )

    ax_ale.set_xlabel(
        f"{feature} ($\\mu g\\ m^{{-3}}$)"
    )

    ax_pdp.set_ylabel(
        f"Predicted {response_name.lower()} ($Mm^{{-1}}$)"
    )

    ax_ale.set_ylabel(
        "Centered accumulated local effect ($Mm^{-1}$)"
    )

    ax_ale.axhline(
        0,
        color="black",
        linestyle="--",
        linewidth=1,
    )

    ax_pdp.grid(alpha=0.25)
    ax_ale.grid(alpha=0.25)

    ax_pdp.legend(frameon=False)
    ax_ale.legend(frameon=False)

plt.tight_layout()
plt.show()


In [ ]:

# ============================================================
# AUDIT 42 — Linear trends in ALE curves
# ============================================================

ale_slope_results = []

for (response_name, feature), model_data in ale_results.items():

    for model_name, ale_df in model_data.items():

        x = ale_df["bin_center"].to_numpy()
        y = ale_df["ALE"].to_numpy()

        # Fit a linear trend to the ALE curve
        slope, intercept = np.polyfit(
            x,
            y,
            1,
        )

        y_pred = intercept + slope*x

        ss_res = np.sum((y - y_pred)**2)
        ss_tot = np.sum((y - np.mean(y))**2)

        r2 = (
            1 - ss_res/ss_tot
            if ss_tot > 0 else np.nan
        )

        ale_slope_results.append({

            "Response": response_name,
            "Predictor": feature,
            "Model": model_name,

            "ALE_trend_slope": slope,
            "ALE_trend_R2": r2,

            "N_bins": len(ale_df),
            "N_observations": ale_df[
                "n_observations"
            ].sum(),

            "Concentration_min": x.min(),
            "Concentration_max": x.max(),

        })


ale_slope_df = pd.DataFrame(
    ale_slope_results
)

display(
    ale_slope_df.round(4)
)

ale_slope_df.to_csv(
    "RF_ALE_slope_results.csv",
    index=False,
)


In [ ]:

# ============================================================
# AUDIT 43 — Relative difference in ALE trend slopes
# ============================================================

ale_comparison = []

for response_name, feature in relationships:

    subset = ale_slope_df[
        (ale_slope_df["Response"] == response_name)
        &
        (ale_slope_df["Predictor"] == feature)
    ]

    full = subset.loc[
        subset["Model"] == "M1_Full",
        "ALE_trend_slope",
    ]

    carbonaceous = subset.loc[
        subset["Model"] == "M5_Carbonaceous",
        "ALE_trend_slope",
    ]

    if len(full) == 0 or len(carbonaceous) == 0:
        continue

    full_slope = full.iloc[0]
    carbon_slope = carbonaceous.iloc[0]

    relative_change = (
        (carbon_slope - full_slope)
        / abs(full_slope) * 100
        if full_slope != 0 else np.nan
    )

    ale_comparison.append({

        "Response": response_name,
        "Predictor": feature,

        "ALE_slope_full": full_slope,
        "ALE_slope_carbonaceous": carbon_slope,

        "Relative_change_percent": relative_change,

    })


ale_comparison_df = pd.DataFrame(
    ale_comparison
)

display(
    ale_comparison_df.round(3)
)

ale_comparison_df.to_csv(
    "RF_ALE_model_comparison.csv",
    index=False,
)


In [ ]:

# ============================================================
# AUDIT 44 — Observational support for ALE intervals
# ============================================================

fig, axes = plt.subplots(
    2, 2,
    figsize=(14, 9),
)

for ax, (response_name, feature) in zip(
    axes.flat,
    relationships,
):

    for model_name in models_to_compare:

        ale_df = ale_results[
            (response_name, feature)
        ][model_name]

        ax.plot(
            ale_df["bin_center"],
            ale_df["n_observations"],
            marker="o",
            linewidth=2,
            label=model_name,
        )

    ax.set_title(
        f"{response_name} vs {feature}"
    )

    ax.set_xlabel(
        f"{feature} ($\\mu g\\ m^{{-3}}$)"
    )

    ax.set_ylabel(
        "Observations per interval"
    )

    ax.grid(alpha=0.25)
    ax.legend(frameon=False)

plt.tight_layout()
plt.show()


In [ ]:

# ============================================================
# AUDIT 45 — Common central-range limits and ALE bins
# ============================================================


models_to_compare = [
    "M1_Full",
    "M5_Carbonaceous",
]

features_to_analyse = ["OA", "rBC"]

n_bins = 20

central_ranges = {}
common_ale_edges = {}

# Use the full model as the reference training dataset
X_reference = rf_models["Scattering"]["M1_Full"]["X_train"]

for feature in features_to_analyse:

    values = X_reference[feature].dropna()

    # Training-set P5 and P95
    p5 = values.quantile(0.05)
    p95 = values.quantile(0.95)

    # Restrict observations to the central range
    central_values = values[
        (values >= p5) &
        (values <= p95)
    ]

    # Quantile-based boundaries within P5-P95
    quantiles = np.linspace(0, 1, n_bins + 1)

    edges = np.quantile(
        central_values,
        quantiles,
    )

    # Remove duplicate boundaries
    edges = np.unique(edges)

    central_ranges[feature] = {
        "p5": p5,
        "p95": p95,
    }

    common_ale_edges[feature] = edges

    print(f"\n{feature}")
    print(f"P5: {p5:.4f}")
    print(f"P95: {p95:.4f}")
    print(f"Number of intervals: {len(edges)-1}")
    print(f"Number of central observations: {len(central_values)}")


In [ ]:

# ============================================================
# AUDIT 46 — Central-range ALE function
# ============================================================

def calculate_central_ale(
    model,
    X,
    feature,
    edges,
    min_samples=5,
):
    """
    Calculate first-order ALE restricted to a predefined
    concentration interval and common bin boundaries.

    Parameters
    ----------
    model : fitted sklearn model
        Random Forest model.

    X : pandas.DataFrame
        Training predictor dataset for this model.

    feature : str
        Predictor being analysed.

    edges : array
        Common bin boundaries derived from M1_Full.

    min_samples : int
        Minimum observations required in each interval.

    Returns
    -------
    ale_df : pandas.DataFrame
        Central-range ALE results.
    """

    X = X.copy()

    X = X.replace(
        [np.inf, -np.inf],
        np.nan,
    )

    X = X.dropna(subset=[feature])

    p5 = edges[0]
    p95 = edges[-1]

    # --------------------------------------------------------
    # 1. Restrict to the common central range
    # --------------------------------------------------------

    X = X[
        (X[feature] >= p5) &
        (X[feature] <= p95)
    ].copy()

    local_effects = []
    bin_counts = []
    bin_centers = []
    bin_lower = []
    bin_upper = []

    # --------------------------------------------------------
    # 2. Calculate local effects in common intervals
    # --------------------------------------------------------

    for i in range(len(edges) - 1):

        lower = edges[i]
        upper = edges[i + 1]

        if i == len(edges) - 2:
            mask = (
                (X[feature] >= lower) &
                (X[feature] <= upper)
            )
        else:
            mask = (
                (X[feature] >= lower) &
                (X[feature] < upper)
            )

        X_bin = X.loc[mask].copy()

        n = len(X_bin)

        if n < min_samples:
            print(
                f"Warning: {feature}, bin {i}: "
                f"only {n} observations."
            )
            continue

        # Modify only the analysed feature
        X_lower = X_bin.copy()
        X_upper = X_bin.copy()

        X_lower[feature] = lower
        X_upper[feature] = upper

        pred_lower = model.predict(X_lower)
        pred_upper = model.predict(X_upper)

        local_effect = np.mean(
            pred_upper - pred_lower
        )

        local_effects.append(local_effect)
        bin_counts.append(n)

        bin_centers.append(
            X_bin[feature].mean()
        )

        bin_lower.append(lower)
        bin_upper.append(upper)

    # --------------------------------------------------------
    # 3. Accumulate local effects
    # --------------------------------------------------------

    local_effects = np.asarray(local_effects)
    bin_counts = np.asarray(bin_counts)

    ale_values = np.cumsum(local_effects)

    # --------------------------------------------------------
    # 4. Center ALE using observation-weighted mean
    # --------------------------------------------------------

    weighted_mean = np.average(
        ale_values,
        weights=bin_counts,
    )

    ale_values = ale_values - weighted_mean

    # --------------------------------------------------------
    # 5. Output
    # --------------------------------------------------------

    ale_df = pd.DataFrame({

        "feature": feature,

        "bin_lower": bin_lower,
        "bin_upper": bin_upper,
        "bin_center": bin_centers,

        "ALE": ale_values,
        "local_effect": local_effects,

        "bin_width": (
            np.asarray(bin_upper) -
            np.asarray(bin_lower)
        ),

        "n_observations": bin_counts,

    })

    return ale_df


In [ ]:

# ============================================================
# AUDIT 47 — Central-range ALE for M1 and M5
# ============================================================

responses = [
    "Scattering",
    "Absorption",
]

ale_central_results = {}

for response_name in responses:

    ale_central_results[response_name] = {}

    for feature in features_to_analyse:

        ale_central_results[
            response_name
        ][feature] = {}

        edges = common_ale_edges[feature]

        for model_name in models_to_compare:

            result = rf_models[
                response_name
            ][model_name]

            model = result["model"]
            X_train = result["X_train"]

            ale_df = calculate_central_ale(
                model=model,
                X=X_train,
                feature=feature,
                edges=edges,
                min_samples=5,
            )

            ale_central_results[
                response_name
            ][feature][model_name] = ale_df

            print(
                f"{response_name} | {feature} | "
                f"{model_name}: "
                f"{len(ale_df)} intervals, "
                f"{ale_df['n_observations'].sum()} observations"
            )

print("\nCentral-range ALE completed.")


In [ ]:

# ============================================================
# AUDIT 48 — Central-range PDP versus ALE
# ============================================================

model_colors = {
    "M1_Full": "tab:blue",
    "M5_Carbonaceous": "tab:orange",
}

relationships = [
    ("Scattering", "OA"),
    ("Scattering", "rBC"),
    ("Absorption", "OA"),
    ("Absorption", "rBC"),
]

fig, axes = plt.subplots(
    4, 2,
    figsize=(15, 18),
)

for row, (response_name, feature) in enumerate(relationships):

    ax_pdp = axes[row, 0]
    ax_ale = axes[row, 1]

    p5 = central_ranges[feature]["p5"]
    p95 = central_ranges[feature]["p95"]

    for model_name in models_to_compare:

        color = model_colors[model_name]

        # ----------------------------------------------------
        # PDP: central range only
        # ----------------------------------------------------

        pdp_data = pdp_spread_results[
            response_name
        ][feature][model_name]

        x = pdp_data["x"]
        y = pdp_data["mean"]
        spread = pdp_data["std"]

        mask = (
            (x >= p5) &
            (x <= p95)
        )

        ax_pdp.plot(
            x[mask],
            y[mask],
            color=color,
            linewidth=2,
            label=model_name,
        )

        ax_pdp.fill_between(
            x[mask],
            y[mask] - spread[mask],
            y[mask] + spread[mask],
            color=color,
            alpha=0.12,
        )

        # ----------------------------------------------------
        # ALE: same central range and common bins
        # ----------------------------------------------------

        ale_df = ale_central_results[
            response_name
        ][feature][model_name]

        ax_ale.plot(
            ale_df["bin_center"],
            ale_df["ALE"],
            marker="o",
            markersize=4,
            linewidth=2,
            color=color,
            label=model_name,
        )

    # --------------------------------------------------------
    # Formatting
    # --------------------------------------------------------

    ax_pdp.set_title(
        f"{response_name} response to {feature} — PDP"
    )

    ax_ale.set_title(
        f"{response_name} response to {feature} — ALE"
    )

    ax_pdp.set_xlim(p5, p95)
    ax_ale.set_xlim(p5, p95)

    ax_pdp.set_xlabel(
        f"{feature} ($\\mu g\\ m^{{-3}}$)"
    )

    ax_ale.set_xlabel(
        f"{feature} ($\\mu g\\ m^{{-3}}$)"
    )

    ax_pdp.set_ylabel(
        f"Predicted {response_name.lower()} ($Mm^{{-1}}$)"
    )

    ax_ale.set_ylabel(
        "Centered accumulated local effect ($Mm^{-1}$)"
    )

    ax_ale.axhline(
        0,
        color="black",
        linestyle="--",
        linewidth=1,
    )

    ax_pdp.grid(alpha=0.25)
    ax_ale.grid(alpha=0.25)

    ax_pdp.legend(frameon=False)
    ax_ale.legend(frameon=False)

plt.tight_layout()
plt.show()


In [ ]:

# ============================================================
# AUDIT 49 — Central-range ALE observational support
# ============================================================

fig, axes = plt.subplots(
    2, 2,
    figsize=(14, 9),
)

for ax, (response_name, feature) in zip(
    axes.flat,
    relationships,
):

    for model_name in models_to_compare:

        ale_df = ale_central_results[
            response_name
        ][feature][model_name]

        ax.plot(
            ale_df["bin_center"],
            ale_df["n_observations"],
            marker="o",
            linewidth=2,
            label=model_name,
        )

    ax.set_title(
        f"{response_name} vs {feature}"
    )

    ax.set_xlabel(
        f"{feature} ($\\mu g\\ m^{{-3}}$)"
    )

    ax.set_ylabel(
        "Observations per interval"
    )

    ax.set_xlim(
        central_ranges[feature]["p5"],
        central_ranges[feature]["p95"],
    )

    ax.grid(alpha=0.25)
    ax.legend(frameon=False)

plt.tight_layout()
plt.show()


# ------------------------------------------------------------
# Bin widths: common boundaries
# ------------------------------------------------------------

fig, axes = plt.subplots(
    1, 2,
    figsize=(13, 4),
)

for ax, feature in zip(
    axes,
    features_to_analyse,
):

    edges = common_ale_edges[feature]

    widths = np.diff(edges)

    ax.bar(
        np.arange(len(widths)),
        widths,
    )

    ax.set_title(
        f"{feature} — Central-range bin widths"
    )

    ax.set_xlabel("ALE interval")
    ax.set_ylabel(
        f"Bin width ($\\mu g\\ m^{{-3}}$)"
    )

    ax.grid(axis="y", alpha=0.25)

plt.tight_layout()
plt.show()


In [ ]:

# ============================================================
# AUDIT 50 — Central-range ALE trend slopes
# ============================================================

ale_central_slope_results = []

for response_name, feature in relationships:

    for model_name in models_to_compare:

        ale_df = ale_central_results[
            response_name
        ][feature][model_name]

        x = ale_df["bin_center"].to_numpy()
        y = ale_df["ALE"].to_numpy()

        # Linear trend
        slope, intercept = np.polyfit(
            x,
            y,
            1,
        )

        y_pred = intercept + slope*x

        ss_res = np.sum(
            (y - y_pred)**2
        )

        ss_tot = np.sum(
            (y - np.mean(y))**2
        )

        r2 = (
            1 - ss_res/ss_tot
            if ss_tot > 0 else np.nan
        )

        ale_central_slope_results.append({

            "Response": response_name,
            "Predictor": feature,
            "Model": model_name,

            "ALE_trend_slope": slope,
            "ALE_trend_intercept": intercept,
            "ALE_trend_R2": r2,

            "N_bins": len(ale_df),

            "N_observations": ale_df[
                "n_observations"
            ].sum(),

            "P5": central_ranges[feature]["p5"],
            "P95": central_ranges[feature]["p95"],

        })


ale_central_slope_df = pd.DataFrame(
    ale_central_slope_results
)

display(
    ale_central_slope_df.round(4)
)

ale_central_slope_df.to_csv(
    "RF_ALE_central_slope_results.csv",
    index=False,
)


In [ ]:

# ============================================================
# AUDIT 51 — Central-range ALE model sensitivity
# ============================================================

ale_central_comparison = []

for response_name, feature in relationships:

    subset = ale_central_slope_df[
        (ale_central_slope_df["Response"] == response_name)
        &
        (ale_central_slope_df["Predictor"] == feature)
    ]

    full_slope = subset.loc[
        subset["Model"] == "M1_Full",
        "ALE_trend_slope",
    ].iloc[0]

    carbon_slope = subset.loc[
        subset["Model"] == "M5_Carbonaceous",
        "ALE_trend_slope",
    ].iloc[0]

    relative_change = (
        (carbon_slope - full_slope)
        / abs(full_slope) * 100
        if full_slope != 0 else np.nan
    )

    ale_central_comparison.append({

        "Response": response_name,
        "Predictor": feature,

        "ALE_slope_full": full_slope,
        "ALE_slope_carbonaceous": carbon_slope,

        "Absolute_change": carbon_slope - full_slope,
        "Relative_change_percent": relative_change,

    })


ale_central_comparison_df = pd.DataFrame(
    ale_central_comparison
)

display(
    ale_central_comparison_df.round(3)
)

ale_central_comparison_df.to_csv(
    "RF_ALE_central_model_comparison.csv",
    index=False,
)


In [ ]:

# ============================================================
# AUDIT 52 — Export central-range ALE curves
# ============================================================

ale_export = []

for response_name, feature in relationships:

    for model_name in models_to_compare:

        ale_df = ale_central_results[
            response_name
        ][feature][model_name].copy()

        ale_df["Response"] = response_name
        ale_df["Model"] = model_name

        ale_export.append(ale_df)


ale_export_df = pd.concat(
    ale_export,
    ignore_index=True,
)

ale_export_df.to_csv(
    "RF_ALE_central_curves.csv",
    index=False,
)

print("Central-range ALE outputs saved.")


In [ ]:
# =========================================================
# FINAL ALE SENSITIVITY ANALYSIS
# Compare 10 and 20 quantile-based bins
# =========================================================

from pathlib import Path

output_dir = Path("ALE_final_sensitivity")
output_dir.mkdir(exist_ok=True)

models_to_compare = [
    "M1_Full",
    "M5_Carbonaceous"
]

features_to_analyse = ["OA", "rBC"]

bin_counts = [10, 20]

PERCENTILE_LOW = 5
PERCENTILE_HIGH = 95

MIN_OBSERVATIONS = 5

# Existing RF response names
response_names = ["Scattering", "Absorption"]

In [ ]:
# =========================================================
# First-order ALE calculation
# =========================================================

def calculate_ale_sensitivity(
    model,
    X,
    feature,
    bin_edges,
    min_observations=5
):

    X = X.copy()

    local_effects = []
    bin_information = []

    for i in range(len(bin_edges) - 1):

        lower = bin_edges[i]
        upper = bin_edges[i + 1]

        if i == len(bin_edges) - 2:
            mask = (
                (X[feature] >= lower) &
                (X[feature] <= upper)
            )
        else:
            mask = (
                (X[feature] >= lower) &
                (X[feature] < upper)
            )

        X_bin = X.loc[mask].copy()

        n_obs = len(X_bin)

        if n_obs < min_observations:
            raise ValueError(
                f"Insufficient observations for {feature}: "
                f"{n_obs} observations in interval "
                f"[{lower:.3f}, {upper:.3f}]"
            )

        X_lower = X_bin.copy()
        X_upper = X_bin.copy()

        X_lower[feature] = lower
        X_upper[feature] = upper

        pred_lower = model.predict(X_lower)
        pred_upper = model.predict(X_upper)

        local_effect = np.mean(
            pred_upper - pred_lower
        )

        bin_width = upper - lower

        local_slope = local_effect / bin_width

        local_effects.append(local_effect)

        bin_information.append({
            "bin_lower": lower,
            "bin_upper": upper,
            "bin_center": (lower + upper) / 2,
            "bin_width": bin_width,
            "local_effect": local_effect,
            "local_slope": local_slope,
            "n_observations": n_obs
        })

    result = pd.DataFrame(bin_information)

    # Accumulated local effects
    result["ALE_uncentered"] = np.cumsum(local_effects)

    # Center the ALE curve using observation-weighted mean
    weighted_mean = np.average(
        result["ALE_uncentered"],
        weights=result["n_observations"]
    )

    result["ALE"] = (
        result["ALE_uncentered"] - weighted_mean
    )

    return result

In [ ]:
# =========================================================
# Calculate ALE for all responses, predictors and models
# =========================================================

ale_sensitivity_results = []

for response_name in response_names:

    print(f"\nProcessing response: {response_name}")

    for feature in features_to_analyse:

        print(f"  Predictor: {feature}")

        # Reference training dataset from M1
        X_reference = rf_models[
            response_name
        ]["M1_Full"]["X_train"].copy()

        # Central concentration limits
        p_low = X_reference[feature].quantile(
            PERCENTILE_LOW / 100
        )

        p_high = X_reference[feature].quantile(
            PERCENTILE_HIGH / 100
        )

        X_reference = X_reference.loc[
            X_reference[feature].between(p_low, p_high)
        ]

        print(
            f"    P5 = {p_low:.3f}; "
            f"P95 = {p_high:.3f}"
        )

        for n_bins in bin_counts:

            # Common quantile boundaries
            quantiles = np.linspace(
                0, 1, n_bins + 1
            )

            bin_edges = np.unique(
                X_reference[feature].quantile(
                    quantiles
                ).values
            )

            actual_bins = len(bin_edges) - 1

            for model_name in models_to_compare:

                model_info = rf_models[
                    response_name
                ][model_name]

                model = model_info["model"]

                predictors = model_info["predictors"]

                X_train = model_info["X_train"].copy()

                # Restrict to the central range
                X_central = X_train.loc[
                    X_train[feature].between(
                        p_low, p_high
                    ),
                    predictors
                ].copy()

                ale_df = calculate_ale_sensitivity(
                    model=model,
                    X=X_central,
                    feature=feature,
                    bin_edges=bin_edges,
                    min_observations=MIN_OBSERVATIONS
                )

                ale_df["Response"] = response_name
                ale_df["Feature"] = feature
                ale_df["Model"] = model_name
                ale_df["N_bins_requested"] = n_bins
                ale_df["N_bins_actual"] = actual_bins

                ale_sensitivity_results.append(ale_df)


ale_sensitivity_df = pd.concat(
    ale_sensitivity_results,
    ignore_index=True
)

ale_sensitivity_df.to_csv(
    output_dir / "ALE_quantile_sensitivity.csv",
    index=False
)

display(ale_sensitivity_df.head())

In [ ]:
# =========================================================
# ALE trend slopes and local slope statistics
# =========================================================

ale_slope_results = []

group_columns = [
    "Response",
    "Feature",
    "Model",
    "N_bins_requested"
]

for keys, group in ale_sensitivity_df.groupby(group_columns):

    response_name, feature, model_name, n_bins = keys

    x = group["bin_center"].to_numpy()
    y = group["ALE"].to_numpy()
    weights = group["n_observations"].to_numpy()

    # Weighted linear regression
    slope, intercept = np.polyfit(
        x,
        y,
        deg=1,
        w=np.sqrt(weights)
    )

    y_pred = slope * x + intercept

    weighted_mean = np.average(
        y,
        weights=weights
    )

    ss_res = np.sum(
        weights * (y - y_pred) ** 2
    )

    ss_tot = np.sum(
        weights * (y - weighted_mean) ** 2
    )

    r_squared = (
        1 - ss_res / ss_tot
        if ss_tot > 0 else np.nan
    )

    local_slopes = group["local_slope"].to_numpy()

    mean_local_slope = np.average(
        local_slopes,
        weights=weights
    )

    ale_slope_results.append({

        "Response": response_name,
        "Feature": feature,
        "Model": model_name,
        "N_bins": n_bins,

        "N_observations": weights.sum(),

        "ALE_trend_slope": slope,
        "ALE_intercept": intercept,
        "ALE_R2": r_squared,

        "Mean_local_slope": mean_local_slope,
        "Median_local_slope": np.median(local_slopes),
        "Std_local_slope": np.std(local_slopes, ddof=1),

        "Min_bin_width": group["bin_width"].min(),
        "Max_bin_width": group["bin_width"].max()
    })


ale_slope_summary = pd.DataFrame(
    ale_slope_results
)

ale_slope_summary.to_csv(
    output_dir / "ALE_slope_summary.csv",
    index=False
)

display(
    ale_slope_summary.round(4)
)

In [ ]:
# =========================================================
# M1 versus M5 model comparison
# =========================================================

model_comparison_results = []

for (response_name, feature, n_bins), group in (
    ale_slope_summary.groupby(
        ["Response", "Feature", "N_bins"]
    )
):

    full = group.loc[
        group["Model"] == "M1_Full"
    ]

    carbonaceous = group.loc[
        group["Model"] == "M5_Carbonaceous"
    ]

    if full.empty or carbonaceous.empty:
        continue

    full = full.iloc[0]
    carbonaceous = carbonaceous.iloc[0]

    slope_full = full["ALE_trend_slope"]
    slope_reduced = carbonaceous["ALE_trend_slope"]

    absolute_change = slope_reduced - slope_full

    relative_change = (
        100 * absolute_change / abs(slope_full)
        if slope_full != 0 else np.nan
    )

    model_comparison_results.append({

        "Response": response_name,
        "Feature": feature,
        "N_bins": n_bins,

        "M1_Full_slope": slope_full,
        "M5_Carbonaceous_slope": slope_reduced,

        "Absolute_change": absolute_change,
        "Relative_change_percent": relative_change,

        "M1_R2": full["ALE_R2"],
        "M5_R2": carbonaceous["ALE_R2"]
    })


ale_model_comparison = pd.DataFrame(
    model_comparison_results
)

ale_model_comparison.to_csv(
    output_dir / "ALE_model_comparison.csv",
    index=False
)

display(
    ale_model_comparison.round(3)
)

In [ ]:
# =========================================================
# Binning sensitivity
# =========================================================

binning_results = []

for (response_name, feature, model_name), group in (
    ale_slope_summary.groupby(
        ["Response", "Feature", "Model"]
    )
):

    result_10 = group.loc[group["N_bins"] == 10]
    result_20 = group.loc[group["N_bins"] == 20]

    if result_10.empty or result_20.empty:
        continue

    slope_10 = result_10.iloc[0]["ALE_trend_slope"]
    slope_20 = result_20.iloc[0]["ALE_trend_slope"]

    absolute_difference = slope_10 - slope_20

    relative_difference = (
        100 * absolute_difference / abs(slope_20)
        if slope_20 != 0 else np.nan
    )

    binning_results.append({

        "Response": response_name,
        "Feature": feature,
        "Model": model_name,

        "Slope_10_bins": slope_10,
        "Slope_20_bins": slope_20,

        "Absolute_difference": absolute_difference,
        "Relative_difference_percent": relative_difference
    })


ale_binning_comparison = pd.DataFrame(
    binning_results
)

ale_binning_comparison.to_csv(
    output_dir / "ALE_binning_sensitivity.csv",
    index=False
)

display(
    ale_binning_comparison.round(3)
)

In [ ]:
# =========================================================
# ALE curve comparison
# =========================================================

fig, axes = plt.subplots(
    2, 2,
    figsize=(14, 10)
)

plot_config = [
    ("Scattering", "OA", axes[0, 0]),
    ("Scattering", "rBC", axes[0, 1]),
    ("Absorption", "OA", axes[1, 0]),
    ("Absorption", "rBC", axes[1, 1])
]

colors = {
    "M1_Full": "tab:blue",
    "M5_Carbonaceous": "tab:orange"
}

linestyles = {
    10: "-",
    20: "--"
}

for response_name, feature, ax in plot_config:

    subset = ale_sensitivity_df[
        (ale_sensitivity_df["Response"] == response_name) &
        (ale_sensitivity_df["Feature"] == feature)
    ]

    for model_name in models_to_compare:

        for n_bins in bin_counts:

            data = subset[
                (subset["Model"] == model_name) &
                (subset["N_bins_requested"] == n_bins)
            ].sort_values("bin_center")

            if data.empty:
                continue

            ax.plot(
                data["bin_center"],
                data["ALE"],
                color=colors[model_name],
                linestyle=linestyles[n_bins],
                marker="o",
                markersize=3,
                label=f"{model_name} — {n_bins} bins"
            )

    ax.axhline(
        0,
        color="black",
        linestyle=":",
        linewidth=1
    )

    ax.set_title(
        f"{response_name} response to {feature} — ALE"
    )

    ax.set_xlabel(
        f"{feature} ($\\mu g\\ m^{{-3}}$)"
    )

    ax.set_ylabel("Centered ALE (Mm$^{-1}$)")

    ax.grid(alpha=0.3)
    ax.legend(fontsize=8)

plt.tight_layout()

plt.savefig(
    output_dir / "ALE_model_binning_comparison.png",
    dpi=300,
    bbox_inches="tight"
)

plt.show()

In [ ]:
# =========================================================
# Local ALE response per unit concentration
# =========================================================

fig, axes = plt.subplots(
    2, 2,
    figsize=(14, 10)
)

for response_name, feature, ax in plot_config:

    subset = ale_sensitivity_df[
        (ale_sensitivity_df["Response"] == response_name) &
        (ale_sensitivity_df["Feature"] == feature)
    ]

    for model_name in models_to_compare:

        for n_bins in bin_counts:

            data = subset[
                (subset["Model"] == model_name) &
                (subset["N_bins_requested"] == n_bins)
            ].sort_values("bin_center")

            if data.empty:
                continue

            ax.plot(
                data["bin_center"],
                data["local_slope"],
                color=colors[model_name],
                linestyle=linestyles[n_bins],
                marker="o",
                markersize=3,
                label=f"{model_name} — {n_bins} bins"
            )

    ax.axhline(
        0,
        color="black",
        linestyle=":",
        linewidth=1
    )

    ax.set_title(
        f"{response_name} local response to {feature}"
    )

    ax.set_xlabel(
        f"{feature} ($\\mu g\\ m^{{-3}}$)"
    )

    ax.set_ylabel(
        "Local ALE slope "
        "(Mm$^{-1}$ / $\\mu g\\ m^{-3}$)"
    )

    ax.grid(alpha=0.3)
    ax.legend(fontsize=8)

plt.tight_layout()

plt.savefig(
    output_dir / "ALE_local_response_comparison.png",
    dpi=300,
    bbox_inches="tight"
)

plt.show()

In [ ]:
# =========================================================
# Bin width and observation-count diagnostics
# =========================================================

fig, axes = plt.subplots(
    2, 2,
    figsize=(14, 9)
)

diagnostics = [
    ("OA", "bin_width", "OA interval width"),
    ("rBC", "bin_width", "rBC interval width"),
    ("OA", "n_observations", "OA observations per interval"),
    ("rBC", "n_observations", "rBC observations per interval")
]

for ax, (feature, variable, title) in zip(
    axes.flat,
    diagnostics
):

    subset = ale_sensitivity_df[
        (ale_sensitivity_df["Feature"] == feature) &
        (ale_sensitivity_df["Response"] == "Scattering") &
        (ale_sensitivity_df["Model"] == "M1_Full")
    ]

    for n_bins, marker, linestyle in [
        (10, "s", "--"),
        (20, "o", "-")
    ]:

        data = subset[
            subset["N_bins_requested"] == n_bins
        ].sort_values("bin_center")

        ax.plot(
            data["bin_center"],
            data[variable],
            marker=marker,
            linestyle=linestyle,
            label=f"{n_bins} bins"
        )

    ax.set_title(title)

    ax.set_xlabel(
        f"{feature} ($\\mu g\\ m^{{-3}}$)"
    )

    ax.set_ylabel(
        variable.replace("_", " ").title()
    )

    ax.grid(alpha=0.3)
    ax.legend()

plt.tight_layout()

plt.savefig(
    output_dir / "ALE_bin_diagnostics.png",
    dpi=300,
    bbox_inches="tight"
)

plt.show()

In [ ]:
# Define publication-style labels for the optical variables. These
# labels are used automatically in the figures generated throughout
# the Random Forest analysis.
target_labels = {
    "Ext": r"$\sigma_{ext}$",
    "TSC_green": r"$\sigma_{sca,\,green}$",
    "TSC_blue": r"$\sigma_{sca,\,blue}$",
    "TSC_red": r"$\sigma_{sca,\,red}$",
    "ABS_PSAP": r"$\sigma_{abs}$",
}

In [ ]:
def run_random_forest(
    df,
    target_variable,
    model_type="reduced",
    test_size=0.2,
    random_state=42,
    n_estimators=600,
):
    """
    Train and evaluate a Random Forest model.

    Parameters
    ----------
    df : pandas.DataFrame
        Input dataframe.
    target_variable : str
        Name of the variable to be predicted.
    model_type : {"full", "reduced"}, default="reduced"
        Feature set used for the model.
    test_size : float, default=0.2
        Fraction of the dataset used for testing.
    random_state : int, default=42
        Random seed.
    n_estimators : int, default=600
        Number of trees in the Random Forest.

    Returns
    -------
    top_features : pandas.Series
        Feature importance ranked in descending order.
    r2 : float
        Coefficient of determination.
    rmse : float
        Root Mean Square Error.
    features : pandas.DataFrame
        Predictor variables.
    X_train : pandas.DataFrame
        Training predictors.
    X_test : pandas.DataFrame
        Testing predictors.
    target_name : str
        Publication-style name of the predicted variable.
    """

    # Define the predictor sets used in the two Random Forest
    # configurations evaluated in this study.
    full_drop = [
        "Ext", "TSC_green", "TSC_blue", "TSC_red",
        "Flight", "ABS_PSAP", "DG", "AS", "AN"
    ]

    reduced_drop = full_drop + [
        "LAT", "LON", "ALT", "CO"
    ]

    drop_columns = {
        "full": full_drop,
        "reduced": reduced_drop,
    }

    # Build the predictor matrix while automatically excluding the
    # selected target variable.
    features = df.drop(
        columns=list(set(drop_columns[model_type] + [target_variable]))
    )

    # Convert optical coefficients from m⁻¹ to Mm⁻¹ to maintain
    # consistency with the units used throughout the manuscript.
    optical_targets = [
        "Ext",
        "TSC_blue",
        "TSC_green",
        "TSC_red",
        "ABS_PSAP",
    ]

    if target_variable in optical_targets:
        target = df[target_variable] * 1e6
    else:
        target = df[target_variable]

    # Retrieve the publication-style label associated with the selected
    # target variable.
    target_name = target_labels.get(target_variable, target_variable)

    # Split the observations into independent training and testing
    # subsets for model evaluation.
    X_train, X_test, y_train, y_test = train_test_split(
        features,
        target,
        test_size=test_size,
        random_state=random_state,
    )

    # Train the Random Forest regression model.
    rf = RandomForestRegressor(
        n_estimators=n_estimators,
        random_state=random_state,
        n_jobs=-1,
    )

    rf.fit(X_train, y_train)

    # Evaluate the model performance using the independent test dataset.
    y_pred = rf.predict(X_test)

    r2 = r2_score(y_test, y_pred)
    rmse = np.sqrt(mean_squared_error(y_test, y_pred))

    # Rank the predictor variables according to their relative importance
    # in the fitted Random Forest model.
    top_features = (
        pd.Series(rf.feature_importances_, index=features.columns)
        .sort_values(ascending=False)
    )

    return (
        rf,
        top_features,
        r2,
        rmse,
        features,
        X_train,
        X_test,
        target_name,
    )

In [ ]:
def plot_rf_importances_and_export_metrics(
    top_features,
    r2,
    rmse,
    features,
    df_clean,
    X_train,
    X_test,
    target_name,
    csv_path="rf_metrics.csv",
    figsize=(7, 4),
    label_fs=13,
    tick_fs=11
):
    """
    Generate a publication-quality feature importance plot and export the
    Random Forest performance metrics to a CSV file.

    Parameters
    ----------
    top_features : pandas.Series
        Feature importance values sorted in descending order.

    r2 : float
        Coefficient of determination.

    rmse : float
        Root Mean Square Error.

    features : pandas.DataFrame
        Predictor variables used to train the Random Forest model.

    df_clean : pandas.DataFrame
        Quality-controlled dataset used in the analysis.

    X_train, X_test : pandas.DataFrame
        Training and testing predictor matrices.

    target_name : str
        Publication-style label of the predicted variable.

    csv_path : str, optional
        Output path for the exported metrics table.

    Returns
    -------
    pandas.DataFrame
        Table containing the model performance metrics.
    """

    # ---------------------------------------------------------
    # Generate the feature importance plot.
    # ---------------------------------------------------------
    fig, ax = plt.subplots(figsize=figsize)

    top_features.plot(
        kind="bar",
        ax=ax,
        edgecolor="black",
        alpha=0.85
    )

    ax.set_ylabel(
        f"Feature Importances ({target_name})",
        fontsize=label_fs
    )

    ax.set_xlabel("", fontsize=label_fs)

    ax.tick_params(
        axis="x",
        labelsize=tick_fs,
        rotation=45
    )

    ax.tick_params(
        axis="y",
        labelsize=tick_fs
    )

    plt.tight_layout()
    plt.show()

    # ---------------------------------------------------------
    # Compile the model performance metrics into a table that
    # can be directly exported for reporting and manuscript
    # preparation.
    # ---------------------------------------------------------
    metrics_df = pd.DataFrame({
        "Metric": [
            "R2",
            "RMSE",
            "n_features",
            "n_measurements",
            "Train size",
            "Test size",
        ],
        "Value": [
            float(r2),
            float(rmse),
            int(features.shape[1]),
            int(len(df_clean)),
            int(len(X_train)),
            int(len(X_test)),
        ]
    })

    # Export the metrics table as a CSV file.
    metrics_df.to_csv(
        csv_path,
        index=False
    )

    return metrics_df

In [ ]:
# Train and evaluate the Random Forest model using the selected target
# variable and predictor configuration.
(
    rf,
    top_features,
    r2,
    rmse,
    features,
    X_train,
    X_test,
    target_name,
) = run_random_forest(
    df=df_clean,
    target_variable="ABS_PSAP",      # e.g. "Ext", "TSC_green", "ABS_PSAP"
    model_type="reduced",       # Options: "full" or "reduced"
)

# Generate the feature importance figure and export the corresponding
# model performance metrics.
metrics_df = plot_rf_importances_and_export_metrics(
    top_features=top_features,
    r2=r2,
    rmse=rmse,
    features=features,
    df_clean=df_clean,
    X_train=X_train,
    X_test=X_test,
    target_name=target_name,
    csv_path="rf_abs_red_metrics.csv",
)

In [ ]:
def plot_manual_pdp_top2_combined_figure(
    rf,
    X_train,
    feat_importances,
    target_name,
    unit_str="(Mm⁻¹)",
    n_grid=60,
    figsize=(16, 10),
    label_fs=14,
    tick_fs=11,
    legend_fs=15,
    title_fs=14,
    line_width=3,
    panel_label_fs=18
):
    """
    Create ONE figure containing 4 panels:

        a) Top-1 feature — Full PDP
        b) Top-1 feature — Zoomed PDP
        c) Top-2 feature — Full PDP
        d) Top-2 feature — Zoomed PDP

    Layout:
    --------
        [ a ] [ b ]
        [ c ] [ d ]

    Zoomed plots are on the right column.

    Includes:
    ----------
    - 5th/95th percentile lines
    - Weighted linear regression
    - Slope uncertainty propagation
    - Panel labels (a,b,c,d)

    Returns
    -------
    fig, axes, results_dict
    """

    import numpy as np
    import matplotlib.pyplot as plt

    # ======================================================
    # PDP computation
    # ======================================================
    def compute_manual_pdp(
        rf,
        X_train,
        feat,
        n_grid=60
    ):

        feat_values = X_train[feat].dropna()

        if feat_values.empty:
            return None

        grid = np.linspace(
            feat_values.min(),
            feat_values.max(),
            n_grid
        )

        p05 = np.percentile(feat_values, 5)
        p95 = np.percentile(feat_values, 95)

        pdp_mean = []
        pdp_std = []

        for val in grid:

            X_temp = X_train.copy()

            X_temp[feat] = val

            preds = rf.predict(X_temp)

            pdp_mean.append(np.mean(preds))
            pdp_std.append(np.std(preds))

        return (
            grid,
            np.asarray(pdp_mean),
            np.asarray(pdp_std),
            p05,
            p95
        )

    # ======================================================
    # Top-2 features
    # ======================================================
    all_features = (
        feat_importances
        .index
        .tolist()
    )

    if len(all_features) < 2:
        raise ValueError(
            "Need at least 2 features."
        )

    top2_features = all_features[:2]

    y_label = (
        f"Predicted {target_name} {unit_str}"
    ).strip()

    # ======================================================
    # Figure setup
    # ======================================================
    fig, axes = plt.subplots(
        2,
        2,
        figsize=figsize,
        constrained_layout=True
    )

    axes = axes.ravel()

    panel_labels = ["a", "b", "c", "d"]

    results_dict = {}

    # ======================================================
    # Loop through features
    # ======================================================
    for i, feat in enumerate(top2_features):

        # ==================================================
        # Compute PDP
        # ==================================================
        result = compute_manual_pdp(
            rf,
            X_train,
            feat,
            n_grid=n_grid
        )

        if result is None:
            continue

        grid, pdp_mean, pdp_std, p05, p95 = result

        # ==================================================
        # Restrict between percentiles
        # ==================================================
        mask = (
            (grid >= p05)
            &
            (grid <= p95)
        )

        grid_zoom = grid[mask]
        pdp_zoom = pdp_mean[mask]
        std_zoom = pdp_std[mask]

        # ==================================================
        # Weighted regression
        # ==================================================
        safe_std = np.where(
            std_zoom == 0,
            1e-12,
            std_zoom
        )

        weights = 1 / (safe_std ** 2)

        fit_coeffs, covariance = np.polyfit(
            grid_zoom,
            pdp_zoom,
            deg=1,
            w=weights,
            cov=True
        )

        slope = fit_coeffs[0]
        intercept = fit_coeffs[1]

        slope_unc = np.sqrt(
            covariance[0, 0]
        )

        fit_line = (
            slope * grid_zoom
            + intercept
        )

        # ==================================================
        # LEFT PANEL — FULL PDP
        # ==================================================
        ax_full = axes[i * 2]

        ax_full.plot(
            grid,
            pdp_mean,
            lw=line_width,
            label="Mean PDP"
        )

        ax_full.fill_between(
            grid,
            pdp_mean - pdp_std,
            pdp_mean + pdp_std,
            alpha=0.25,
            label="±1σ"
        )

        ax_full.axvline(
            p05,
            linestyle="--",
            linewidth=2,
            alpha=0.9,
            label="5th percentile"
        )

        ax_full.axvline(
            p95,
            linestyle="--",
            linewidth=2,
            alpha=0.9,
            label="95th percentile"
        )

        ax_full.set_xlabel(
            feat + r' ($\mu g\ m^{-3}$)',
            fontsize=label_fs
        )

        ax_full.set_ylabel(
            y_label,
            fontsize=label_fs
        )

        ax_full.tick_params(
            axis="both",
            labelsize=tick_fs
        )

        ax_full.legend(
            fontsize=legend_fs,
            frameon=False,
            loc="lower right"
        )

        ax_full.grid(
            True,
            axis="y",
            linestyle="--",
            alpha=0.5
        )

        ax_full.grid(False, axis="x")

        ax_full.spines["top"].set_visible(False)
        ax_full.spines["right"].set_visible(False)

        # Panel label
        ax_full.text(
            0.02,
            0.96,
            panel_labels[i * 2],
            transform=ax_full.transAxes,
            fontsize=panel_label_fs,
            fontweight="bold",
            va="top"
        )

        # ==================================================
        # RIGHT PANEL — ZOOMED PDP
        # ==================================================
        ax_zoom = axes[i * 2 + 1]

        ax_zoom.plot(
            grid_zoom,
            pdp_zoom,
            lw=line_width,
            label="Mean PDP"
        )

        ax_zoom.fill_between(
            grid_zoom,
            pdp_zoom - std_zoom,
            pdp_zoom + std_zoom,
            alpha=0.25,
            label="±1σ"
        )

        ax_zoom.plot(
            grid_zoom,
            fit_line,
            linestyle="--",
            linewidth=2.5,
            label=(
                f"Slope = "
                f"{slope:.2f} ± "
                f"{slope_unc:.2f}"
            )
        )

        ax_zoom.set_xlim(
            p05,
            p95
        )

        ax_zoom.set_xlabel(
            feat + r' ($\mu g\ m^{-3}$)',
            fontsize=label_fs
        )

        ax_zoom.set_ylabel(
            y_label,
            fontsize=label_fs
        )

        ax_zoom.tick_params(
            axis="both",
            labelsize=tick_fs
        )

        ax_zoom.legend(
            fontsize=legend_fs,
            frameon=False,
            loc="lower right"
        )

        ax_zoom.grid(
            True,
            axis="y",
            linestyle="--",
            alpha=0.5
        )

        ax_zoom.grid(False, axis="x")

        ax_zoom.spines["top"].set_visible(False)
        ax_zoom.spines["right"].set_visible(False)

        # Panel label
        ax_zoom.text(
            0.02,
            0.96,
            panel_labels[i * 2 + 1],
            transform=ax_zoom.transAxes,
            fontsize=panel_label_fs,
            fontweight="bold",
            va="top"
        )

        # ==================================================
        # Save results
        # ==================================================
        results_dict[feat] = {

            "slope": slope,
            "slope_uncertainty": slope_unc,
            "intercept": intercept,
            "p05": p05,
            "p95": p95
        }

    plt.show()

    return fig, axes, results_dict

In [ ]:
# Select the predictors to be analyzed through Partial Dependence
# Profiles (PDPs). Here, two contributors identified by
# the Random Forest model are examined in detail.
filtered_feat_importances = top_features[["OA", "rBC"]]

# Generate the publication-quality PDP figure for the selected
# predictors and retrieve the fitted regression parameters.
fig, axes, results = (
    plot_manual_pdp_top2_combined_figure(
        rf=rf,
        X_train=X_train,
        feat_importances=filtered_feat_importances,
        target_name=target_name,
    )
)

In [ ]:
def run_random_forest_v2(
    df,
    target_variable,
    model_type="reduced",
    test_size=0.2,
    random_state=42,
    n_estimators=600,
):
    """
    Train and evaluate a Random Forest model.

    Parameters
    ----------
    df : pandas.DataFrame
        Input dataframe.
    target_variable : str
        Name of the variable to be predicted.
    model_type : {"full", "reduced"}, default="reduced"
        Feature set used for the model.
    test_size : float, default=0.2
        Fraction of the dataset used for testing.
    random_state : int, default=42
        Random seed.
    n_estimators : int, default=600
        Number of trees in the Random Forest.

    Returns
    -------
    top_features : pandas.Series
        Feature importance ranked in descending order.
    r2 : float
        Coefficient of determination.
    rmse : float
        Root Mean Square Error.
    features : pandas.DataFrame
        Predictor variables.
    X_train : pandas.DataFrame
        Training predictors.
    X_test : pandas.DataFrame
        Testing predictors.
    target_name : str
        Publication-style name of the predicted variable.
    """

    # Define the predictor sets used in the two Random Forest
    # configurations evaluated in this study.
    full_drop = [
        "Ext", "TSC_green", "TSC_blue", "TSC_red",
        "Flight", "ABS_PSAP", "DG", "AS", "AN"
    ]

    reduced_drop = full_drop + [
        "LAT", "LON", "ALT", "CO"
    ]

    drop_columns = {
        "full": full_drop,
        "reduced": reduced_drop,
    }

    # Build the predictor matrix while automatically excluding the
    # selected target variable.
    features = df.drop(
        columns=list(set(drop_columns[model_type] + [target_variable]))
    )

    # Convert optical coefficients from m⁻¹ to Mm⁻¹ to maintain
    # consistency with the units used throughout the manuscript.
    optical_targets = [
        "Ext",
        "TSC_blue",
        "TSC_green",
        "TSC_red",
        "ABS_PSAP",
    ]

    if target_variable in optical_targets:
        target = df[target_variable] * 1e6
    else:
        target = df[target_variable]

    # Retrieve the publication-style label associated with the selected
    # target variable.
    target_name = target_labels.get(target_variable, target_variable)

    # Split the observations into independent training and testing
    # subsets for model evaluation.
    X_train, X_test, y_train, y_test = train_test_split(
        features,
        target,
        test_size=test_size,
        random_state=random_state,
    )

    # Train the Random Forest regression model.
    rf = RandomForestRegressor(
        n_estimators=n_estimators,
        random_state=random_state,
        n_jobs=-1,
    )

    rf.fit(X_train, y_train)

    # Evaluate the model performance using the independent test dataset.
    y_pred = rf.predict(X_test)

    r2 = r2_score(y_test, y_pred)
    rmse = np.sqrt(mean_squared_error(y_test, y_pred))

    # Rank the predictor variables according to their relative importance
    # in the fitted Random Forest model.
    feature_importance = pd.Series(
        rf.feature_importances_,
        index=features.columns
    )

    return (
        rf,
        feature_importance,
        r2,
        rmse,
        features,
        X_train,
        X_test,
        target_name,
    )

In [ ]:
def plot_rf_feature_importance_comparison(
    results,
    figsize=(8, 8),
    label_fs=12,
    tick_fs=11,
):

    import matplotlib.pyplot as plt
    import pandas as pd

    # --------------------------------------------------
    # Feature order (same for every model)
    # --------------------------------------------------
    feature_order = [
        "OA",
        "rBC",
        "SO4",
        "NH4",
        "NO3",
        "CO",
        "PM_coarse",
        "ALT",
        "LAT",
        "LON",
    ]

    # --------------------------------------------------
    # Consistent colors
    # --------------------------------------------------
    feature_colors = {
        "OA": "green",
        "rBC": "black",
        "SO4": "red",
        "NH4": "orange",
        "NO3": "navy",
        "CO": "mediumpurple",
        "PM_coarse": "blue",
        "ALT": "dimgray",
        "LAT": "gray",
        "LON": "silver",
    }
    # --------------------------------------------------
    # Collect feature importance from each model
    # --------------------------------------------------
    importance_list = []
    model_names = []

    for result in results:

        (
            rf,
            feature_importance,
            r2,
            rmse,
            features,
            X_train,
            X_test,
            target_name,
        ) = result

        # Reorder features
        feature_importance = (
            feature_importance
            .reindex(feature_order)
            .fillna(0)
        )

        importance_list.append(feature_importance)
        model_names.append(target_name)

    # --------------------------------------------------
    # Common x-axis limit
    # --------------------------------------------------
    max_importance = max(
        imp.max() for imp in importance_list
    ) * 1.10

    # --------------------------------------------------
    # Create figure
    # --------------------------------------------------
    fig, axes = plt.subplots(
        nrows=3,
        ncols=1,
        figsize=figsize,
        sharex=True
    )

    panel_titles = [
        "(a) Scattering",
        "(b) Absorption",
        "(c) Extinction"
    ]

    # --------------------------------------------------
    # Plot each RF model
    # --------------------------------------------------
    for ax, imp, panel_title in zip(
        axes,
        importance_list,
        panel_titles
    ):

        colors = [
            feature_colors[f]
            for f in feature_order
        ]

        ax.bar(
            feature_order,
            imp.values,
            color=colors,
            edgecolor="black",
            linewidth=0.8,
            alpha=0.9,
        )

        ax.set_ylim(0, max_importance)

        ax.set_ylabel(
            "Importance",
            fontsize=label_fs
        )

        ax.set_title(
            panel_title,
            fontsize=label_fs,
            loc="left",
            fontweight="bold",
        )

        # Remove unnecessary borders
        ax.spines["top"].set_visible(False)
        ax.spines["right"].set_visible(False)

        # Tick size
        ax.tick_params(
            axis="y",
            labelsize=tick_fs
        )

        # Group separators
        ax.axvline(
            4.5,
            color="0.5",
            ls="--",
            lw=1,
        )

        #ax.axvline(
        #    5.5,
        #    color="0.5",
        #    ls="--",
        #    lw=1,
        #)

        #ax.axvline(
        #    6.5,
        #    color="0.5",
        #    ls="--",
        #    lw=1,
        #)

        # Group labels
        ymax = max_importance * 1.01

        ax.text(
            2,
            ymax,
            "Mass composition",
            ha="center",
            va="top",
            fontsize=10,
        )

        #ax.text(
        #    5,
        #    ymax,
        #    "Tracer",
        #    ha="center",
        #    va="top",
        #    fontsize=10,
        #)

        #ax.text(
        #    6,
        #    ymax,
        #    "Coarse",
        #    ha="center",
        #    va="top",
        #    fontsize=10,
        #)

        ax.text(
            7,
            ymax,
            "Context",
            ha="center",
            va="top",
            fontsize=10,
        )
    # --------------------------------------------------
    # Bottom x-axis
    # --------------------------------------------------
    axes[-1].tick_params(
        axis="x",
        rotation=45,
        labelsize=tick_fs,
    )

    # Hide x labels from upper panels
    for ax in axes[:-1]:
        ax.tick_params(
            axis="x",
            labelbottom=False
        )

    axes[-1].set_xlabel(
        "Predictor",
        fontsize=label_fs
    )

    plt.tight_layout()

    return fig, axes

In [ ]:
def plot_rf_feature_importance_comparison_v2(
    results,
    figsize=(11, 5),
    label_fs=12,
    tick_fs=11,
):

    import matplotlib.pyplot as plt

    # --------------------------------------------------
    # Feature order
    # --------------------------------------------------
    feature_order = [
        "OA",
        "rBC",
        "SO4",
        "NH4",
        "NO3",
        "PM_coarse",
    ]

    # --------------------------------------------------
    # Colors
    # --------------------------------------------------
    feature_colors = {
        "OA": "green",
        "rBC": "black",
        "SO4": "red",
        "NH4": "orange",
        "NO3": "navy",
        "CO": "mediumpurple",
        "PM_coarse": "blue",
        "ALT": "dimgray",
        "LAT": "gray",
        "LON": "silver",
    }

    # --------------------------------------------------
    # Collect importances
    # --------------------------------------------------
    importance_list = []

    for result in results:

        (
            rf,
            feature_importance,
            r2,
            rmse,
            features,
            X_train,
            X_test,
            target_name,
        ) = result

        feature_importance = (
            feature_importance
            .reindex(feature_order)
            .fillna(0)
        )

        importance_list.append(feature_importance)

    # --------------------------------------------------
    # Common x limit
    # --------------------------------------------------
    max_importance = (
        max(imp.max() for imp in importance_list) * 1.10
    )

    # --------------------------------------------------
    # Figure
    # --------------------------------------------------
    fig, axes = plt.subplots(
        1,
        3,
        figsize=figsize,
        sharex=True,
        sharey=True,
    )

    panel_titles = [
        "(a) Scattering",
        "(b) Absorption",
        "(c) Extinction",
    ]

    y = range(len(feature_order))

    # --------------------------------------------------
    # Plot
    # --------------------------------------------------
    for i, (ax, imp, title) in enumerate(
        zip(axes, importance_list, panel_titles)
    ):

        colors = [
            feature_colors[f]
            for f in feature_order
        ]

        ax.barh(
            y,
            imp.values,
            color=colors,
            edgecolor="black",
            linewidth=0.8,
        )

        ax.set_xlim(0, max_importance)

        ax.set_title(
            title,
            loc="left",
            fontsize=label_fs,
            fontweight="bold",
        )

        ax.spines["top"].set_visible(False)
        ax.spines["right"].set_visible(False)

        ax.tick_params(
            axis="x",
            labelsize=tick_fs,
        )

        # Hide predictor labels from all panels
        ax.set_yticks(y)
        ax.tick_params(
            axis="y",
            left=False,
            labelleft=False,
        )


        ax.set_xlabel(
            "Importance",
            fontsize=label_fs,
        )

    # Keep OA at the top
    axes[0].invert_yaxis()

    legend_handles = [
    Patch(
        facecolor=feature_colors[f],
        edgecolor="black",
        label=f
    )
    for f in feature_order
    ]

    fig.legend(
        handles=legend_handles,
        loc="center left",
        bbox_to_anchor=(0.90, 0.5),
        frameon=False,
        fontsize=tick_fs,
        title="Predictor",
        title_fontsize=label_fs,
    )

    plt.tight_layout(rect=[0, 0, 0.90, 1])

    return fig, axes

In [ ]:
results = []

for target in [
    "TSC_green",
    "ABS_PSAP",
    "Ext",
]:

    results.append(
        run_random_forest_v2(
            df=df_clean,
            target_variable=target,
            model_type="reduced",
        )
    )

plot_rf_feature_importance_comparison_v2(results)